# Analyse des contrats de location de box

## Contexte du projet

Cette étude porte sur les données contractuelles de quatre centres de self-stockage, anonymisés sous les noms de Centre A, Centre B, Centre C et Centre S.

Le jeu de données consolidé comprend **9 904 contrats**. Il provient initialement de huit fichiers : un historique complet et un fichier des contrats en cours pour chacun des quatre centres.

## Objectifs

- Centraliser et fiabiliser les données des quatre centres.
- Analyser le portefeuille de contrats et l’activité commerciale.
- Étudier les annulations intervenant avant le démarrage de la location.
- Identifier les profils associés à un départ précoce dans les quatre premiers mois.
- Analyser la fidélisation des clients sur une période de douze mois.
- Préparer l’étude du risque d’impayé.
- Produire des indicateurs utiles au pilotage dans Power BI.

## Périmètre et limites

Les données analysées décrivent les contrats, les surfaces louées, les montants contractuels, les remises, les services et les statuts.

Elles ne contiennent pas le référentiel complet des box disponibles et occupés. Le taux d’occupation et la rentabilité réelle ne peuvent donc pas être calculés avec les seules données disponibles.

# 1. Importation des bibliothèques

In [3]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# 2. Chargement du fichier consolidé

Cette étape charge le fichier regroupant les contrats des quatre centres. Un premier contrôle permet de vérifier ses dimensions et d’afficher un aperçu des données.

In [4]:
chemin_fichier = (
    r"C:\Users\anais\OneDrive\Desktop\location de box"
    r"\Regroupement 4 centres\regroupement_centre.csv"
)

regroupement_centre = pd.read_csv(
    chemin_fichier,
    encoding="utf-8-sig",
    low_memory=False
)

print("Dimensions du fichier :", regroupement_centre.shape)


Dimensions du fichier : (9904, 38)


# 3. Contrôle de la qualité des données

Cette section vérifie la structure du fichier consolidé, l’unicité des identifiants, les doublons, les valeurs manquantes et le format des différentes variables.

## 3.1 Dimensions et structure

In [5]:
print("Nombre de lignes :", regroupement_centre.shape[0])
print("Nombre de colonnes :", regroupement_centre.shape[1])

regroupement_centre.info()

Nombre de lignes : 9904
Nombre de colonnes : 38
<class 'pandas.DataFrame'>
RangeIndex: 9904 entries, 0 to 9903
Data columns (total 38 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Id                                  9904 non-null   str    
 1   S.No                                9904 non-null   int64  
 2   Deal initiated by                   9448 non-null   str    
 3   Deal closed by                      9448 non-null   str    
 4   Payment mode                        9900 non-null   str    
 5   Tax User Type                       9904 non-null   str    
 6   Storage                             9904 non-null   str    
 7   Property                            9904 non-null   str    
 8   Contract Id                         9904 non-null   str    
 9   Contract Status                     9904 non-null   str    
 10  Rent starts on                      9904 non-null   str    
 11  Recurr

In [6]:
structure_colonnes = pd.DataFrame({
    "Type": regroupement_centre.dtypes.astype(str),
    "Valeurs renseignées": regroupement_centre.notna().sum(),
    "Valeurs manquantes": regroupement_centre.isna().sum(),
    "Pourcentage manquant": (
        regroupement_centre.isna().mean() * 100
    ).round(2),
    "Valeurs distinctes": regroupement_centre.nunique(dropna=True)
})

structure_colonnes

,Type,Valeurs renseignées,Valeurs manquantes,Pourcentage manquant,Valeurs distinctes
Id,str,9904,0,0.00,9904
S.No,int64,9904,0,0.00,9904
Deal initiated by,str,9448,456,4.60,35
Deal closed by,str,9448,456,4.60,35
Payment mode,str,9900,4,0.04,6
Tax User Type,str,9904,0,0.00,4
Storage,str,9904,0,0.00,1
Property,str,9904,0,0.00,4
Contract Id,str,9904,0,0.00,9861
Contract Status,str,9904,0,0.00,7


## 3.2 Contrôle des identifiants et des doublons

La colonne `Id_centre` combine le centre et l’identifiant du contrat. Elle doit identifier chaque ligne de manière unique dans le fichier consolidé.

In [7]:
controle_identifiants = pd.Series({
    "Nombre de lignes": len(regroupement_centre),
    "Identifiants uniques": regroupement_centre["Id_centre"].nunique(),
    "Identifiants manquants": regroupement_centre["Id_centre"].isna().sum(),
    "Identifiants dupliqués": regroupement_centre["Id_centre"].duplicated().sum(),
    "Lignes entièrement dupliquées": regroupement_centre.duplicated().sum()
})

controle_identifiants

Nombre de lignes                 9904
Identifiants uniques             9904
Identifiants manquants              0
Identifiants dupliqués              0
Lignes entièrement dupliquées       0
dtype: int64

### Conclusion du contrôle

Chaque contrat dispose d’un identifiant `Id_centre` unique et renseigné. Aucun doublon complet ni aucun doublon sur cet identifiant n’a été détecté. Le fichier consolidé contient donc une ligne unique par contrat.

## 3.3 Analyse des valeurs manquantes

Les valeurs manquantes sont examinées avant tout traitement. Leur absence peut provenir d’une anomalie, mais aussi du fonctionnement normal du contrat.

Par exemple, l’absence de date de départ est logique pour un contrat toujours en cours. Les valeurs ne seront donc pas remplacées ou supprimées automatiquement.

In [8]:
controle_valeurs_manquantes = pd.DataFrame({
    "Type": regroupement_centre.dtypes.astype(str),
    "Valeurs renseignées": regroupement_centre.notna().sum(),
    "Valeurs manquantes": regroupement_centre.isna().sum(),
    "Pourcentage manquant (%)": (
        regroupement_centre.isna().mean() * 100
    ).round(2),
    "Valeurs distinctes": regroupement_centre.nunique(dropna=True)
})

controle_valeurs_manquantes = controle_valeurs_manquantes.sort_values(
    by="Pourcentage manquant (%)",
    ascending=False
)

controle_valeurs_manquantes

,Type,Valeurs renseignées,Valeurs manquantes,Pourcentage manquant (%),Valeurs distinctes
New Insurance Premium,float64,0,9904,100.00,0
New Rent Price,float64,0,9904,100.00,0
Service,str,3174,6730,67.95,24
Moveout recorded on,str,4974,4930,49.78,1074
Moveout scheduled on,str,5101,4803,48.50,1328
Moveout date,str,6070,3834,38.71,1245
Discount starts on,str,7748,2156,21.77,1444
Discount ends on,str,7748,2156,21.77,2094
Rent starts on.1,str,9078,826,8.34,55
Deal closed by,str,9448,456,4.60,35


In [9]:
colonnes_avec_valeurs_manquantes = controle_valeurs_manquantes[
    controle_valeurs_manquantes["Valeurs manquantes"] > 0
]

colonnes_avec_valeurs_manquantes

,Type,Valeurs renseignées,Valeurs manquantes,Pourcentage manquant (%),Valeurs distinctes
New Insurance Premium,float64,0,9904,100.00,0
New Rent Price,float64,0,9904,100.00,0
Service,str,3174,6730,67.95,24
Moveout recorded on,str,4974,4930,49.78,1074
Moveout scheduled on,str,5101,4803,48.50,1328
Moveout date,str,6070,3834,38.71,1245
Discount starts on,str,7748,2156,21.77,1444
Discount ends on,str,7748,2156,21.77,2094
Rent starts on.1,str,9078,826,8.34,55
Deal closed by,str,9448,456,4.60,35


### Première interprétation

Les valeurs manquantes ne doivent pas toutes recevoir le même traitement :

- l’absence de date de départ est normale pour un contrat qui n’est pas terminé ;
- l’absence de remise ou de service peut signifier qu’aucun avantage ou service n’a été souscrit ;
- certaines informations commerciales peuvent ne pas avoir été renseignées ;
- les colonnes entièrement vides doivent être contrôlées avant leur éventuelle suppression.

Les traitements seront donc réalisés selon la signification métier de chaque variable.

### 3.3.1 Colonnes entièrement vides

Une colonne ne contenant aucune valeur exploitable n’apporte aucune information à l’analyse et ne pourra pas être utilisée dans les modèles.

In [10]:
colonnes_entierement_vides = regroupement_centre.columns[
    regroupement_centre.isna().all()
].tolist()

print("Nombre de colonnes entièrement vides :", len(colonnes_entierement_vides))
print("Colonnes concernées :", colonnes_entierement_vides)

Nombre de colonnes entièrement vides : 2
Colonnes concernées : ['New Rent Price', 'New Insurance Premium']


In [11]:
regroupement_centre = regroupement_centre.drop(
    columns=colonnes_entierement_vides
)

print(
    "Nouvelles dimensions du fichier :",
    regroupement_centre.shape
)

Nouvelles dimensions du fichier : (9904, 36)


### Traitement appliqué

Les colonnes `New Rent Price` et `New Insurance Premium` étaient entièrement vides. Elles ont été retirées du jeu de données, car elles ne pouvaient contribuer ni à l’analyse descriptive ni aux modèles prédictifs.

Les autres valeurs manquantes sont conservées à ce stade afin d’être traitées selon leur signification métier.

In [12]:
colonnes_dates = [
    "Rent starts on",
    "Discount starts on",
    "Discount ends on",
    "Invoice generated till",
    "Moveout scheduled on",
    "Moveout recorded on",
    "Moveout date"
]

valeurs_avant_conversion = (
    regroupement_centre[colonnes_dates]
    .notna()
    .sum()
)

for colonne in colonnes_dates:
    regroupement_centre[colonne] = pd.to_datetime(
        regroupement_centre[colonne],
        format="mixed",
        yearfirst=True,
        errors="coerce"
    )

controle_dates = pd.DataFrame({
    "Valeurs avant conversion": valeurs_avant_conversion,
    "Dates valides après conversion":
        regroupement_centre[colonnes_dates].notna().sum()
})

controle_dates["Valeurs non converties"] = (
    controle_dates["Valeurs avant conversion"]
    - controle_dates["Dates valides après conversion"]
)

controle_dates

,Valeurs avant conversion,Dates valides après conversion,Valeurs non converties
Rent starts on,9904,9904,0
Discount starts on,7748,7748,0
Discount ends on,7748,7748,0
Invoice generated till,9904,9904,0
Moveout scheduled on,5101,5101,0
Moveout recorded on,4974,4974,0
Moveout date,6070,6070,0


In [13]:
regroupement_centre[colonnes_dates].dtypes

Rent starts on            datetime64[us]
Discount starts on        datetime64[us]
Discount ends on          datetime64[us]
Invoice generated till    datetime64[us]
Moveout scheduled on      datetime64[us]
Moveout recorded on       datetime64[us]
Moveout date              datetime64[us]
dtype: object

### Résultat du contrôle

Les colonnes temporelles ont été converties au format date. Les valeurs absentes sont conservées lorsqu’elles correspondent à une situation métier normale, notamment pour les contrats ne disposant pas encore d’une date de départ.

Ces dates permettront ensuite de calculer les durées de location et de construire les populations observables pour les analyses à quatre et douze mois.

## 3.5 Conversion et contrôle des variables numériques

Les surfaces, montants, pourcentages et durées doivent rester au format numérique pour permettre les calculs, les visualisations Power BI et la modélisation.

Les valeurs impossibles à convertir sont temporairement remplacées par une valeur manquante afin d’être identifiées.

In [14]:
colonnes_numeriques = [
    "Area",
    "Volume",
    "Standard Taxable Amount",
    "Contract Taxable Amount",
    "Contract Taxable Discounted Amount",
    "Taxable amount",
    "Discount (%)",
    "Discount duration",
    "Insurance Premium",
    "Service price"
]

valeurs_avant_conversion = {
    colonne: regroupement_centre[colonne].notna().sum()
    for colonne in colonnes_numeriques
}

for colonne in colonnes_numeriques:
    regroupement_centre[colonne] = pd.to_numeric(
        regroupement_centre[colonne],
        errors="coerce"
    )

controle_conversion_numerique = pd.DataFrame({
    "Valeurs avant conversion": [
        valeurs_avant_conversion[colonne]
        for colonne in colonnes_numeriques
    ],
    "Valeurs numériques après conversion": [
        regroupement_centre[colonne].notna().sum()
        for colonne in colonnes_numeriques
    ]
}, index=colonnes_numeriques)

controle_conversion_numerique["Valeurs non converties"] = (
    controle_conversion_numerique["Valeurs avant conversion"]
    - controle_conversion_numerique[
        "Valeurs numériques après conversion"
    ]
)

controle_conversion_numerique

,Valeurs avant conversion,Valeurs numériques après conversion,Valeurs non converties
Area,9904,9904,0
Volume,9904,9904,0
Standard Taxable Amount,9904,9904,0
Contract Taxable Amount,9904,9904,0
Contract Taxable Discounted Amount,9904,9904,0
Taxable amount,9904,9904,0
Discount (%),9904,9904,0
Discount duration,9904,9904,0
Insurance Premium,9904,9904,0
Service price,9904,9904,0


In [15]:
regroupement_centre[colonnes_numeriques].dtypes

Area                                  float64
Volume                                float64
Standard Taxable Amount               float64
Contract Taxable Amount               float64
Contract Taxable Discounted Amount    float64
Taxable amount                        float64
Discount (%)                          float64
Discount duration                     float64
Insurance Premium                     float64
Service price                           int64
dtype: object

In [16]:
regroupement_centre[colonnes_numeriques].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
Area,9904.0,5.19,4.49,0.93,2.04,4.00,7.00,45.00
Volume,9904.0,13.92,12.42,0.93,5.60,10.80,17.76,126.00
Standard Taxable Amount,9904.0,108.77,68.02,0.00,65.83,99.17,129.17,638.25
Contract Taxable Amount,9904.0,110.23,70.22,0.00,65.83,99.17,132.50,700.52
Contract Taxable Discounted Amount,9904.0,91.88,64.87,0.00,51.92,74.25,114.66,700.52
Taxable amount,9904.0,96.52,64.63,0.00,57.50,87.28,122.87,671.93
Discount (%),9904.0,27.43,21.47,0.00,10.00,30.00,40.00,100.00
Discount duration,9904.0,16.85,34.05,0.00,2.00,6.00,12.00,289.00
Insurance Premium,9904.0,9.06,5.87,0.00,7.00,9.00,9.90,77.00
Service price,9904.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00


### Résultat du contrôle

Les variables nécessaires aux calculs ont été converties au format numérique. Elles pourront être utilisées pour analyser les surfaces, les montants contractuels, les remises, les assurances et les services.

La conversion ne suffit cependant pas à garantir la cohérence métier des valeurs. Les valeurs négatives, nulles ou exceptionnellement élevées seront contrôlées dans l’étape suivante.

## 3.6 Contrôle de la cohérence des variables numériques

Cette étape recherche les valeurs négatives, nulles ou exceptionnellement élevées. Une valeur atypique n’est pas nécessairement erronée : elle doit être interprétée selon la signification métier de la colonne avant toute correction.

In [17]:
controle_coherence_numerique = pd.DataFrame({
    "Minimum": regroupement_centre[colonnes_numeriques].min(),
    "Médiane": regroupement_centre[colonnes_numeriques].median(),
    "Moyenne": regroupement_centre[colonnes_numeriques].mean(),
    "Maximum": regroupement_centre[colonnes_numeriques].max(),
    "Valeurs négatives": [
        (regroupement_centre[colonne] < 0).sum()
        for colonne in colonnes_numeriques
    ],
    "Valeurs nulles": [
        (regroupement_centre[colonne] == 0).sum()
        for colonne in colonnes_numeriques
    ]
})

controle_coherence_numerique.round(2)

,Minimum,Médiane,Moyenne,Maximum,Valeurs négatives,Valeurs nulles
Area,0.93,4.00,5.19,45.00,0,0
Volume,0.93,10.80,13.92,126.00,0,0
Standard Taxable Amount,0.00,99.17,108.77,638.25,0,15
Contract Taxable Amount,0.00,99.17,110.23,700.52,0,95
Contract Taxable Discounted Amount,0.00,74.25,91.88,700.52,0,96
Taxable amount,0.00,87.28,96.52,671.93,0,797
Discount (%),0.00,30.00,27.43,100.00,0,2177
Discount duration,0.00,6.00,16.85,289.00,0,2156
Insurance Premium,0.00,9.00,9.06,77.00,0,405
Service price,0.00,0.00,0.00,0.00,0,9904


In [18]:
surfaces_invalides = regroupement_centre[
    regroupement_centre["Area"].isna()
    | (regroupement_centre["Area"] <= 0)
]

print(
    "Nombre de surfaces manquantes ou non positives :",
    len(surfaces_invalides)
)

surfaces_invalides[
    ["Id_centre", "Centre", "Area", "Volume"]
].head(20)

Nombre de surfaces manquantes ou non positives : 0


,Id_centre,Centre,Area,Volume


In [19]:
remises_invalides = regroupement_centre[
    regroupement_centre["Discount (%)"].notna()
    & (
        (regroupement_centre["Discount (%)"] < 0)
        | (regroupement_centre["Discount (%)"] > 100)
    )
]

print(
    "Nombre de pourcentages de remise hors de l’intervalle 0–100 :",
    len(remises_invalides)
)

remises_invalides[
    ["Id_centre", "Centre", "Discount (%)", "Discount"]
].head(20)

Nombre de pourcentages de remise hors de l’intervalle 0–100 : 0


,Id_centre,Centre,Discount (%),Discount


In [20]:
colonnes_montants = [
    "Standard Taxable Amount",
    "Contract Taxable Amount",
    "Contract Taxable Discounted Amount",
    "Taxable amount",
    "Insurance Premium",
    "Service price"
]

montants_negatifs = pd.Series({
    colonne: (regroupement_centre[colonne] < 0).sum()
    for colonne in colonnes_montants
})

montants_negatifs

Standard Taxable Amount               0
Contract Taxable Amount               0
Contract Taxable Discounted Amount    0
Taxable amount                        0
Insurance Premium                     0
Service price                         0
dtype: int64

### Règles d’interprétation

- Une surface nulle ou négative serait incompatible avec une location de box et nécessiterait une vérification.
- Un pourcentage de remise doit normalement être compris entre 0 et 100.
- Une remise ou une prime d’assurance égale à zéro peut être légitime.
- Un montant contractuel différent du montant standard peut correspondre à une remise ou à une modification tarifaire.
- La colonne `Service price` doit être interprétée avec prudence si elle contient uniquement des zéros, car le prix réel du service pourrait être inclus dans une autre colonne ou ne pas avoir été exporté.
- Les valeurs atypiques ne sont pas supprimées sans validation métier.

## 3.7 Contrôle des variables catégorielles

Cette étape examine les principales catégories utilisées dans l’analyse. Elle permet de repérer les différences d’écriture, les catégories rares et les éventuelles valeurs incohérentes avant de construire les indicateurs et les modèles prédictifs.

In [ ]:
colonnes_categorielles = [
    "Centre",
    "Contract Status",
    "Payment mode",
    "Tax User Type",
    "Recurring Period",
    "Storage"
]

for colonne in colonnes_categorielles:
    print(f"\n===== {colonne} =====")
    print(
        regroupement_centre[colonne]
        .value_counts(dropna=False)
    )

In [22]:
statuts_contrats = (
    regroupement_centre["Contract Status"]
    .value_counts(dropna=False)
    .rename_axis("Statut")
    .reset_index(name="Nombre de contrats")
)

statuts_contrats["Part (%)"] = (
    statuts_contrats["Nombre de contrats"]
    / len(regroupement_centre)
    * 100
).round(2)

statuts_contrats

,Statut,Nombre de contrats,Part (%)
0,Lease Closed,6065,61.24
1,Active,2375,23.98
2,Lease Cancelled,1295,13.08
3,Notice Period,57,0.58
4,Access Revoked,51,0.51
5,Initiated,43,0.43
6,Over locked,18,0.18


In [23]:
controle_statut_en_cours = pd.crosstab(
    regroupement_centre["Contract Status"],
    regroupement_centre["Est_en_cours"],
    margins=True
)

controle_statut_en_cours

Est_en_cours,False,True,All
Contract Status,,,
Access Revoked,0,51,51
Active,0,2375,2375
Initiated,0,43,43
Lease Cancelled,1295,0,1295
Lease Closed,6065,0,6065
Notice Period,0,57,57
Over locked,0,18,18
All,7360,2544,9904


In [24]:
for colonne in colonnes_categorielles:
    if regroupement_centre[colonne].dtype == "object":
        valeurs_avec_espaces = (
            regroupement_centre[colonne]
            .dropna()
            .astype(str)
            .loc[
                lambda serie:
                serie.ne(serie.str.strip())
            ]
        )

        print(
            f"{colonne} :",
            len(valeurs_avec_espaces),
            "valeur(s) avec espaces en début ou fin"
        )

### Interprétation métier des statuts

Les statuts ne représentent pas tous la même situation :

- `Active` correspond à une location normalement active ;
- `Initiated` correspond à un contrat en préparation ;
- `Notice Period` signale un départ annoncé ;
- `Access Revoked` correspond à un impayé encore potentiellement récupérable ;
- `Over locked` correspond à une situation d’impayé plus critique ;
- `Lease Closed` correspond à une location terminée ;
- `Lease Cancelled` correspond à un contrat annulé avant la prise effective du box.

Cette distinction sera conservée dans les analyses afin de ne pas confondre une fin normale, une annulation avant location, un départ annoncé et une situation d’impayé.

## 3.8 Cohérence entre les statuts et les dates

Les dates doivent être cohérentes avec la situation du contrat. Ce contrôle permet notamment de vérifier :

- qu’un contrat terminé possède une date de départ ;
- qu’un contrat annulé avant location n’est pas assimilé à un départ ;
- qu’un contrat actif ne possède pas une ancienne date de départ ;
- que la date de départ n’est pas antérieure au début de la location.

In [25]:
regroupement_centre["Duree_location_jours"] = (
    regroupement_centre["Moveout date"]
    - regroupement_centre["Rent starts on"]
).dt.days

regroupement_centre[
    [
        "Rent starts on",
        "Moveout date",
        "Duree_location_jours"
    ]
].head()

,Rent starts on,Moveout date,Duree_location_jours
0,2025-02-02,2026-06-01,484.0
1,2025-02-13,NaT,NaN
2,2025-03-01,NaT,NaN
3,2025-01-30,2025-05-14,104.0
4,2025-02-14,NaT,NaN


In [26]:
contrats_termines_sans_depart = regroupement_centre[
    (regroupement_centre["Contract Status"] == "Lease Closed")
    & (regroupement_centre["Moveout date"].isna())
]

print(
    "Contrats terminés sans date de départ :",
    len(contrats_termines_sans_depart)
)

contrats_termines_sans_depart[
    [
        "Id_centre",
        "Centre",
        "Contract Status",
        "Rent starts on",
        "Moveout scheduled on",
        "Moveout recorded on",
        "Moveout date"
    ]
].head(20)

Contrats terminés sans date de départ : 0


,Id_centre,Centre,Contract Status,Rent starts on,Moveout scheduled on,Moveout recorded on,Moveout date


In [ ]:
contrats_actifs_avec_depart = regroupement_centre[
    (regroupement_centre["Contract Status"] == "Active")
    & (regroupement_centre["Moveout date"].notna())
]

print(
    "Contrats actifs avec une date de départ :",
    len(contrats_actifs_avec_depart)
)

contrats_actifs_avec_depart[
    [
        "Id_centre",
        "Centre",
        "Contract Status",
        "Rent starts on",
        "Moveout scheduled on",
        "Moveout recorded on",
        "Moveout date",
        "Duree_location_jours"
    ]
]

In [ ]:
durees_negatives = regroupement_centre[
    regroupement_centre["Duree_location_jours"] < 0
].sort_values(
    by="Duree_location_jours"
)

print("Durées négatives :", len(durees_negatives))

durees_negatives[
    [
        "Id_centre",
        "Centre",
        "Contract Status",
        "Rent starts on",
        "Moveout date",
        "Duree_location_jours"
    ]
].head(20)

In [29]:
synthese_coherence_dates = pd.Series({
    "Contrats terminés sans date de départ":
        len(contrats_termines_sans_depart),

    "Contrats actifs avec une date de départ":
        len(contrats_actifs_avec_depart),

    "Durées de location négatives":
        len(durees_negatives),

    "Durées de location calculables":
        regroupement_centre["Duree_location_jours"].notna().sum()
})

synthese_coherence_dates

Contrats terminés sans date de départ         0
Contrats actifs avec une date de départ       3
Durées de location négatives                 70
Durées de location calculables             6070
dtype: int64

### 3.9 Analyse des durées négatives

In [30]:
contrats_duree_negative = regroupement_centre.loc[
    regroupement_centre["Duree_location_jours"] < 0,
    [
        "Id_centre",
        "Centre",
        "Contract Status",
        "Rent starts on",
        "Moveout date",
        "Duree_location_jours"
    ]
].sort_values("Duree_location_jours")

print(
    "Nombre de contrats avec une durée négative :",
    len(contrats_duree_negative)
)

Nombre de contrats avec une durée négative : 70


In [ ]:
print("Répartition par centre :")
display(
    contrats_duree_negative["Centre"]
    .value_counts()
    .rename("Nombre d'anomalies")
)

print("\nRépartition par statut :")
display(
    contrats_duree_negative["Contract Status"]
    .value_counts()
    .rename("Nombre d'anomalies")
)

#### Traitement retenu

Les 70 durées négatives concernent exclusivement des contrats terminés. La majorité présente un faible décalage, mais la date de départ reste antérieure au début de location et ne permet donc pas de calculer une durée fiable.

Les contrats sont conservés dans la base afin de ne pas perdre leurs autres informations. Seule la durée de location négative est remplacée par une valeur manquante. Ces lignes seront ainsi exclues automatiquement des analyses portant sur les durées.

In [32]:
masque_duree_negative = (
    regroupement_centre["Duree_location_jours"] < 0
)

regroupement_centre.loc[
    masque_duree_negative,
    "Duree_location_jours"
] = pd.NA

print(
    "Durées négatives restantes :",
    (
        regroupement_centre["Duree_location_jours"] < 0
    ).sum()
)

print(
    "Durées exploitables restantes :",
    regroupement_centre["Duree_location_jours"].notna().sum()
)

Durées négatives restantes : 0
Durées exploitables restantes : 6000


### 3.10 Contrats actifs possédant une date de départ
Un contrat au statut « Active » ne devrait normalement pas posséder de date de départ définitive. Ces contrats sont isolés afin de déterminer si la date correspond à un départ réel, à une ancienne information ou à une incohérence de mise à jour.

In [ ]:
contrats_actifs_avec_depart = regroupement_centre.loc[
    (regroupement_centre["Contract Status"] == "Active")
    & regroupement_centre["Moveout date"].notna(),
    [
        "Id_centre",
        "Centre",
        "Contract Status",
        "Rent starts on",
        "Moveout scheduled on",
        "Moveout recorded on",
        "Moveout date",
        "Duree_location_jours"
    ]
]

print(
    "Nombre de contrats actifs avec une date de départ :",
    len(contrats_actifs_avec_depart)
)

contrats_actifs_avec_depart

#### Traitement retenu

Trois contrats actifs possèdent une date de départ, alors qu’aucune date de départ planifiée ou enregistrée n’est renseignée. Le statut actuel « Active » est retenu comme information de référence.

Afin de préserver les données sources, la colonne `Moveout date` n’est pas modifiée. Une colonne dédiée aux analyses est créée et la date de départ y est neutralisée pour ces trois contrats. Un indicateur permet également de conserver la trace de cette anomalie.

In [34]:
# Identifier les contrats actifs ayant une date de départ incohérente
regroupement_centre["Anomalie_actif_avec_depart"] = (
    (regroupement_centre["Contract Status"] == "Active")
    & regroupement_centre["Moveout date"].notna()
)

# Conserver la date d'origine et créer une date dédiée aux analyses
regroupement_centre["Moveout_date_analyse"] = (
    regroupement_centre["Moveout date"].copy()
)

regroupement_centre.loc[
    regroupement_centre["Anomalie_actif_avec_depart"],
    "Moveout_date_analyse"
] = pd.NaT

print(
    "Anomalies conservées pour traçabilité :",
    regroupement_centre["Anomalie_actif_avec_depart"].sum()
)

print(
    "Contrats actifs avec une date de départ exploitable :",
    (
        (regroupement_centre["Contract Status"] == "Active")
        & regroupement_centre["Moveout_date_analyse"].notna()
    ).sum()
)

Anomalies conservées pour traçabilité : 3
Contrats actifs avec une date de départ exploitable : 0


### 3.11 Création de la durée de location définitive

La durée définitive est calculée uniquement pour les contrats possédant des dates de début et de départ cohérentes. Les contrats actifs avec une date de départ incohérente et les durées négatives sont exclus de cet indicateur, sans supprimer les contrats de la base.

In [35]:
regroupement_centre["Duree_location_analyse_jours"] = (
    regroupement_centre["Moveout_date_analyse"]
    - regroupement_centre["Rent starts on"]
).dt.days

# Neutraliser les durées négatives
regroupement_centre.loc[
    regroupement_centre["Duree_location_analyse_jours"] < 0,
    "Duree_location_analyse_jours"
] = pd.NA

controle_duree_definitive = pd.Series({
    "Durées exploitables":
        regroupement_centre["Duree_location_analyse_jours"]
        .notna()
        .sum(),

    "Durées négatives restantes":
        (
            regroupement_centre["Duree_location_analyse_jours"] < 0
        ).sum(),

    "Contrats actifs avec une durée calculée":
        (
            (regroupement_centre["Contract Status"] == "Active")
            & regroupement_centre[
                "Duree_location_analyse_jours"
            ].notna()
        ).sum()
})

controle_duree_definitive

Durées exploitables                        5997
Durées négatives restantes                    0
Contrats actifs avec une durée calculée       0
dtype: int64

## 4. Définition des problématiques métier

L’analyse prédictive est organisée autour de trois problématiques distinctes, car elles concernent des populations et des actions métier différentes.

### 4.1 Annulation avant le début de la location

L’objectif est d’identifier, dès la création du contrat, les dossiers présentant un risque d’annulation avant l’entrée effective du client dans le box.

- Cas positif : `Lease Cancelled`
- Cas négatif : contrat ayant effectivement commencé
- Population : ensemble des contrats, sous réserve de disposer des informations connues avant le démarrage

### 4.2 Départ après le début de la location

L’objectif est d’identifier les caractéristiques associées aux locations courtes et d’estimer le risque de départ après 4, 6 et 12 mois.

Les trois horizons seront étudiés séparément :

- départ dans les 4 mois ;
- départ dans les 6 mois ;
- départ dans les 12 mois.

Une location terminée avant l’horizon constitue un cas positif. Un contrat encore présent après l’horizon constitue un cas négatif. Les contrats n’ayant pas bénéficié d’une durée d’observation suffisante seront exclus du modèle correspondant.

### 4.3 Risque d’impayé

L’objectif est d’identifier les contrats associés à une situation d’impayé.

- `Access Revoked` : situation potentiellement récupérable ;
- `Over locked` : situation plus critique ;
- population : contrats ayant effectivement commencé.

Ce troisième sujet sera traité avec prudence, car les cas d’impayé sont peu nombreux et les données disponibles ne contiennent pas le détail des factures, des échéances et des règlements.

## 5. Création des cibles métier

### 5.1 Annulation avant le début de la location

La population comprend :

- les contrats annulés avant leur démarrage (`Lease Cancelled`) ;
- les contrats dont la location a effectivement commencé.

Les contrats au statut `Initiated` sont exclus : ils sont encore en préparation et leur issue n’est pas connue.

La cible vaut :

- `1` : contrat annulé avant location ;
- `0` : location effectivement commencée.

In [36]:
# Statuts correspondant à une location ayant effectivement commencé
statuts_location_commencee = [
    "Active",
    "Lease Closed",
    "Notice Period",
    "Access Revoked",
    "Over locked"
]

# Conserver uniquement les contrats dont l'issue est connue
population_annulation = regroupement_centre.loc[
    regroupement_centre["Contract Status"].isin(
        ["Lease Cancelled"] + statuts_location_commencee
    )
].copy()

# Créer la cible binaire
population_annulation["Cible_annulation"] = (
    population_annulation["Contract Status"]
    .eq("Lease Cancelled")
    .astype(int)
)

# Créer un libellé lisible
population_annulation["Libelle_annulation"] = (
    population_annulation["Cible_annulation"]
    .map({
        0: "Location commencée",
        1: "Annulation avant location"
    })
)

print(
    "Population exploitable :",
    len(population_annulation)
)

print("\nContrats exclus au statut Initiated :")
print(
    (regroupement_centre["Contract Status"] == "Initiated").sum()
)

print("\nRépartition de la cible :")
display(
    population_annulation["Libelle_annulation"]
    .value_counts()
)

print("\nRépartition en pourcentage :")
display(
    (
        population_annulation["Libelle_annulation"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)

Population exploitable : 9861

Contrats exclus au statut Initiated :
43

Répartition de la cible :


Libelle_annulation
Location commencée           8566
Annulation avant location    1295
Name: count, dtype: int64


Répartition en pourcentage :


Libelle_annulation
Location commencée           86.87
Annulation avant location    13.13
Name: proportion, dtype: float64

### 5.2 Population des locations commencées

Les analyses du départ et de l’impayé portent uniquement sur les contrats dont la location a effectivement commencé.

Les contrats annulés avant location (`Lease Cancelled`) et les contrats encore en préparation (`Initiated`) sont donc exclus.

In [37]:
population_location = regroupement_centre.loc[
    regroupement_centre["Contract Status"]
    .isin(statuts_location_commencee)
].copy()

print(
    "Nombre de contrats ayant effectivement commencé :",
    len(population_location)
)

print("\nRépartition des statuts :")
display(
    population_location["Contract Status"]
    .value_counts()
)

print("\nContrôle des exclusions :")
print(
    "Lease Cancelled conservés :",
    (
        population_location["Contract Status"]
        == "Lease Cancelled"
    ).sum()
)

print(
    "Initiated conservés :",
    (
        population_location["Contract Status"]
        == "Initiated"
    ).sum()
)

Nombre de contrats ayant effectivement commencé : 8566

Répartition des statuts :


Contract Status
Lease Closed      6065
Active            2375
Notice Period       57
Access Revoked      51
Over locked         18
Name: count, dtype: int64


Contrôle des exclusions :
Lease Cancelled conservés : 0
Initiated conservés : 0


### 5.3 Détermination de la date d’observation

La création des cibles de départ nécessite une date de référence correspondant à la fin de la période couverte par les données.

Cette date permet de distinguer :

- les contrats suffisamment anciens pour être évalués ;
- les contrats trop récents, dont le devenir à 4, 6 ou 12 mois n’est pas encore observable.

Les dates de départ planifiées ne sont pas utilisées pour définir cette limite, car elles peuvent se situer dans le futur.

In [38]:
colonnes_controle_observation = [
    "Rent starts on",
    "Invoice generated till",
    "Moveout recorded on",
    "Moveout_date_analyse"
]

dates_maximales = pd.Series({
    colonne: regroupement_centre[colonne].max()
    for colonne in colonnes_controle_observation
})

print("Dates maximales disponibles :")
display(dates_maximales)

Dates maximales disponibles :


Rent starts on           2026-08-22
Invoice generated till   2027-03-31
Moveout recorded on      2026-07-29
Moveout_date_analyse     2026-08-11
dtype: datetime64[us]

In [39]:
# Date correspondant à la dernière extraction des données
date_observation = pd.Timestamp("2026-07-31")

print(
    "Date d’observation retenue :",
    date_observation.strftime("%d/%m/%Y")
)

Date d’observation retenue : 31/07/2026


La date maximale de la colonne `Invoice generated till` atteint mars 2027, car elle contient des échéances de facturation futures. Elle ne représente donc pas la date de mise à jour de la base.

La date d’observation est fixée au **31 juillet 2026**, correspondant à la dernière extraction disponible. Les informations postérieures à cette date ne seront pas utilisées pour déterminer si un départ a déjà eu lieu.

### 5.4 Création des cibles de départ à 4, 6 et 12 mois

Trois horizons sont étudiés afin de distinguer les départs rapides des locations plus durables :

- départ dans les 4 premiers mois ;
- départ dans les 6 premiers mois ;
- départ dans les 12 premiers mois.

Pour chaque horizon :

- la cible vaut `1` lorsque le départ a eu lieu avant la date limite ;
- la cible vaut `0` lorsque le contrat a dépassé cette date limite sans partir ;
- les contrats trop récents sont exclus, car leur devenir n’est pas encore observable ;
- les contrats terminés dont la durée est incohérente sont également exclus.

In [40]:
def creer_population_depart(donnees, horizon_mois, date_observation):
    population = donnees.copy()

    # Date à laquelle chaque contrat atteint l’horizon étudié
    population["Date_limite_horizon"] = (
        population["Rent starts on"]
        + pd.DateOffset(months=horizon_mois)
    )

    # Contrats terminés dont la durée ne peut pas être fiabilisée
    anomalie_duree = (
        (population["Contract Status"] == "Lease Closed")
        & population["Duree_location_analyse_jours"].isna()
    )

    # Départ réellement observé avant la date d’observation
    depart_observe = (
        (population["Contract Status"] == "Lease Closed")
        & population["Moveout_date_analyse"].notna()
        & (
            population["Moveout_date_analyse"]
            <= date_observation
        )
    )

    # Départ réalisé dans l’horizon étudié
    depart_dans_horizon = (
        depart_observe
        & (
            population["Moveout_date_analyse"]
            <= population["Date_limite_horizon"]
        )
    )

    # Contrat ayant eu assez de recul pour atteindre l’horizon
    horizon_entierement_observe = (
        population["Date_limite_horizon"]
        <= date_observation
    )

    # Une ligne est exploitable si :
    # - le départ a déjà été observé dans l’horizon ;
    # - ou le contrat a atteint l’horizon sans partir avant celui-ci.
    ligne_exploitable = (
        (depart_dans_horizon | horizon_entierement_observe)
        & ~anomalie_duree
    )

    population = population.loc[ligne_exploitable].copy()

    population[f"Cible_depart_{horizon_mois}_mois"] = (
        depart_dans_horizon.loc[population.index]
        .astype(int)
    )

    return population

In [41]:
population_depart_4_mois = creer_population_depart(
    population_location,
    horizon_mois=4,
    date_observation=date_observation
)

population_depart_6_mois = creer_population_depart(
    population_location,
    horizon_mois=6,
    date_observation=date_observation
)

population_depart_12_mois = creer_population_depart(
    population_location,
    horizon_mois=12,
    date_observation=date_observation
)

In [42]:
controle_populations_depart = pd.DataFrame({
    "Horizon": ["4 mois", "6 mois", "12 mois"],
    "Population exploitable": [
        len(population_depart_4_mois),
        len(population_depart_6_mois),
        len(population_depart_12_mois)
    ],
    "Nombre de départs": [
        population_depart_4_mois[
            "Cible_depart_4_mois"
        ].sum(),
        population_depart_6_mois[
            "Cible_depart_6_mois"
        ].sum(),
        population_depart_12_mois[
            "Cible_depart_12_mois"
        ].sum()
    ]
})

controle_populations_depart["Taux de départ (%)"] = (
    controle_populations_depart["Nombre de départs"]
    / controle_populations_depart["Population exploitable"]
    * 100
).round(2)

controle_populations_depart

,Horizon,Population exploitable,Nombre de départs,Taux de départ (%)
0,4 mois,7914,2850,36.01
1,6 mois,7737,3506,45.31
2,12 mois,7271,4659,64.08


### 5.5 Création de la cible d’impayé

L’analyse des impayés porte uniquement sur les contrats dont la location a effectivement commencé.

La cible distingue :

- `1` : impayé identifié, correspondant aux statuts `Access Revoked` ou `Over locked` ;
- `0` : aucun impayé identifié dans les statuts disponibles.

Les deux situations positives ne présentent pas le même niveau de gravité :

- `Access Revoked` peut encore correspondre à une situation récupérable ;
- `Over locked` représente une situation plus critique.

Cette cible indique uniquement la présence d’un statut associé à un impayé. Elle ne mesure ni son montant, ni son ancienneté, ni les paiements déjà reçus.

In [43]:
population_impaye = population_location.copy()

population_impaye["Cible_impaye"] = (
    population_impaye["Contract Status"]
    .isin(["Access Revoked", "Over locked"])
    .astype(int)
)

population_impaye["Libelle_impaye"] = (
    population_impaye["Cible_impaye"]
    .map({
        0: "Aucun impayé identifié",
        1: "Impayé identifié"
    })
)

print(
    "Population exploitable :",
    len(population_impaye)
)

print("\nRépartition de la cible :")
display(
    population_impaye["Libelle_impaye"]
    .value_counts()
)

print("\nRépartition en pourcentage :")
display(
    (
        population_impaye["Libelle_impaye"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)

print("\nDétail des statuts d’impayé :")
display(
    population_impaye.loc[
        population_impaye["Cible_impaye"] == 1,
        "Contract Status"
    ].value_counts()
)

Population exploitable : 8566

Répartition de la cible :


Libelle_impaye
Aucun impayé identifié    8497
Impayé identifié            69
Name: count, dtype: int64


Répartition en pourcentage :


Libelle_impaye
Aucun impayé identifié    99.19
Impayé identifié           0.81
Name: proportion, dtype: float64


Détail des statuts d’impayé :


Contract Status
Access Revoked    51
Over locked       18
Name: count, dtype: int64

### 5.6 Synthèse des populations et des cibles

Ce tableau récapitule les volumes disponibles pour chaque problématique avant la préparation des variables explicatives.

Il permet notamment d’identifier le niveau de déséquilibre des classes et d’adapter la méthode d’évaluation de chaque modèle.

In [44]:
synthese_cibles = pd.DataFrame({
    "Sujet": [
        "Annulation avant location",
        "Départ dans les 4 mois",
        "Départ dans les 6 mois",
        "Départ dans les 12 mois",
        "Impayé identifié"
    ],

    "Population exploitable": [
        len(population_annulation),
        len(population_depart_4_mois),
        len(population_depart_6_mois),
        len(population_depart_12_mois),
        len(population_impaye)
    ],

    "Cas positifs": [
        population_annulation["Cible_annulation"].sum(),
        population_depart_4_mois["Cible_depart_4_mois"].sum(),
        population_depart_6_mois["Cible_depart_6_mois"].sum(),
        population_depart_12_mois["Cible_depart_12_mois"].sum(),
        population_impaye["Cible_impaye"].sum()
    ]
})

synthese_cibles["Cas négatifs"] = (
    synthese_cibles["Population exploitable"]
    - synthese_cibles["Cas positifs"]
)

synthese_cibles["Taux positif (%)"] = (
    synthese_cibles["Cas positifs"]
    / synthese_cibles["Population exploitable"]
    * 100
).round(2)

synthese_cibles

,Sujet,Population exploitable,Cas positifs,Cas négatifs,Taux positif (%)
0,Annulation avant location,9861,1295,8566,13.13
1,Départ dans les 4 mois,7914,2850,5064,36.01
2,Départ dans les 6 mois,7737,3506,4231,45.31
3,Départ dans les 12 mois,7271,4659,2612,64.08
4,Impayé identifié,8566,69,8497,0.81


## 6. Préparation des variables explicatives

### 6.1 Prévention des fuites de données

Une fuite de données apparaît lorsqu’un modèle utilise une information qui ne serait pas encore connue au moment où la prédiction doit être réalisée.

Les modèles utiliseront uniquement les caractéristiques connues lors de la création ou du démarrage du contrat :

- le centre ;
- le mode de paiement ;
- le type de client ;
- la périodicité ;
- la surface ;
- les caractéristiques tarifaires ;
- la remise ;
- l’assurance ;
- les services souscrits ;
- la période prévue de début de location.

Les informations suivantes sont exclues des variables explicatives :

- `Contract Status`, car il sert à définir certaines cibles ;
- les dates de départ ;
- la durée réelle de location ;
- les indicateurs d’anomalie créés lors du nettoyage ;
- les identifiants ;
- les informations de facturation postérieures au démarrage.

Cette séparation garantit que le modèle ne connaît pas indirectement l’

In [45]:
variables_sources_candidates = [
    "Centre",
    "Payment mode",
    "Tax User Type",
    "Recurring Period",
    "Area",
    "Standard Taxable Amount",
    "Contract Taxable Amount",
    "Contract Taxable Discounted Amount",
    "Discount (%)",
    "Discount duration",
    "Insurance Premium",
    "Service",
    "Rent starts on"
]

controle_variables_sources = pd.DataFrame({
    "Type": regroupement_centre[
        variables_sources_candidates
    ].dtypes.astype(str),

    "Valeurs manquantes": regroupement_centre[
        variables_sources_candidates
    ].isna().sum(),

    "Pourcentage manquant": (
        regroupement_centre[
            variables_sources_candidates
        ].isna().mean() * 100
    ).round(2),

    "Valeurs distinctes": regroupement_centre[
        variables_sources_candidates
    ].nunique(dropna=True)
})

controle_variables_sources

,Type,Valeurs manquantes,Pourcentage manquant,Valeurs distinctes
Centre,str,0,0.00,4
Payment mode,str,4,0.04,6
Tax User Type,str,0,0.00,4
Recurring Period,str,1,0.01,3
Area,float64,0,0.00,280
Standard Taxable Amount,float64,0,0.00,172
Contract Taxable Amount,float64,0,0.00,1308
Contract Taxable Discounted Amount,float64,0,0.00,2220
Discount (%),float64,0,0.00,20
Discount duration,float64,0,0.00,48


### 6.2 Création des variables explicatives

Les variables sont préparées à partir des informations disponibles au début du contrat.

Afin d’éviter la redondance entre les différents montants contractuels :

- la surface est conservée ;
- le prix standard au m² est calculé ;
- le pourcentage et la durée de remise sont conservés ;
- un indicateur précise si une remise est appliquée ;
- un indicateur précise si un service complémentaire est souscrit.

Le mois de début est également conservé afin d’étudier un éventuel effet saisonnier. L’année n’est pas utilisée comme caractéristique principale, car elle pourrait capter l’évolution historique des tarifs ou de la base plutôt qu’un véritable profil client.

In [46]:
# Créer une base dédiée aux variables explicatives
donnees_modelisation = regroupement_centre.copy()

# Valeurs catégorielles très rarement manquantes
donnees_modelisation["Payment mode"] = (
    donnees_modelisation["Payment mode"]
    .fillna("Non renseigné")
)

donnees_modelisation["Recurring Period"] = (
    donnees_modelisation["Recurring Period"]
    .fillna("Non renseigné")
)

# Prix standard rapporté à la surface du box
donnees_modelisation["Prix_standard_m2"] = (
    donnees_modelisation["Standard Taxable Amount"]
    / donnees_modelisation["Area"]
)

# Présence d’une remise
donnees_modelisation["A_remise"] = (
    donnees_modelisation["Discount (%)"] > 0
).astype(int)

# Présence d’un service complémentaire
donnees_modelisation["A_service"] = (
    donnees_modelisation["Service"].notna()
).astype(int)

# Caractéristiques temporelles connues au démarrage
donnees_modelisation["Mois_debut"] = (
    donnees_modelisation["Rent starts on"].dt.month
)

donnees_modelisation["Trimestre_debut"] = (
    donnees_modelisation["Rent starts on"].dt.quarter
)

In [47]:
variables_explicatives = [
    "Centre",
    "Payment mode",
    "Tax User Type",
    "Recurring Period",
    "Area",
    "Prix_standard_m2",
    "Discount (%)",
    "Discount duration",
    "Insurance Premium",
    "A_remise",
    "A_service",
    "Mois_debut"
]

controle_variables_explicatives = pd.DataFrame({
    "Type": donnees_modelisation[
        variables_explicatives
    ].dtypes.astype(str),

    "Valeurs manquantes": donnees_modelisation[
        variables_explicatives
    ].isna().sum(),

    "Valeurs distinctes": donnees_modelisation[
        variables_explicatives
    ].nunique(dropna=True)
})

controle_variables_explicatives

,Type,Valeurs manquantes,Valeurs distinctes
Centre,str,0,4
Payment mode,str,0,7
Tax User Type,str,0,4
Recurring Period,str,0,4
Area,float64,0,280
Prix_standard_m2,float64,0,527
Discount (%),float64,0,20
Discount duration,float64,0,48
Insurance Premium,float64,0,64
A_remise,int64,0,2


### 6.3 Sélection définitive des variables

La variable `A_remise` n’est pas retenue dans les modèles, car elle est directement calculée à partir de `Discount (%)`. Conserver les deux variables introduirait une information redondante.

La sélection finale comprend donc :

- quatre variables catégorielles ;
- sept variables numériques ou binaires.

Cette même base de variables sera utilisée pour comparer les différents modèles et les différents horizons de départ.

In [48]:
variables_categorielles = [
    "Centre",
    "Payment mode",
    "Tax User Type",
    "Recurring Period"
]

variables_numeriques = [
    "Area",
    "Prix_standard_m2",
    "Discount (%)",
    "Discount duration",
    "Insurance Premium",
    "A_service",
    "Mois_debut"
]

variables_modele = (
    variables_categorielles
    + variables_numeriques
)

print("Nombre de variables sources :", len(variables_modele))
print("\nVariables catégorielles :")
print(variables_categorielles)

print("\nVariables numériques ou binaires :")
print(variables_numeriques)

Nombre de variables sources : 11

Variables catégorielles :
['Centre', 'Payment mode', 'Tax User Type', 'Recurring Period']

Variables numériques ou binaires :
['Area', 'Prix_standard_m2', 'Discount (%)', 'Discount duration', 'Insurance Premium', 'A_service', 'Mois_debut']


### 6.4 Association des variables aux différentes populations
Comme les cibles ont été créées dans plusieurs DataFrames, nous allons récupérer les variables grâce à l’index des contrats :

In [49]:
X_annulation = donnees_modelisation.loc[
    population_annulation.index,
    variables_modele
].copy()

y_annulation = population_annulation[
    "Cible_annulation"
].copy()

X_depart_4_mois = donnees_modelisation.loc[
    population_depart_4_mois.index,
    variables_modele
].copy()

y_depart_4_mois = population_depart_4_mois[
    "Cible_depart_4_mois"
].copy()

X_depart_6_mois = donnees_modelisation.loc[
    population_depart_6_mois.index,
    variables_modele
].copy()

y_depart_6_mois = population_depart_6_mois[
    "Cible_depart_6_mois"
].copy()

X_depart_12_mois = donnees_modelisation.loc[
    population_depart_12_mois.index,
    variables_modele
].copy()

y_depart_12_mois = population_depart_12_mois[
    "Cible_depart_12_mois"
].copy()

X_impaye = donnees_modelisation.loc[
    population_impaye.index,
    variables_modele
].copy()

y_impaye = population_impaye[
    "Cible_impaye"
].copy()

In [50]:
controle_modelisation = pd.DataFrame({
    "Sujet": [
        "Annulation",
        "Départ 4 mois",
        "Départ 6 mois",
        "Départ 12 mois",
        "Impayé"
    ],
    "Lignes dans X": [
        len(X_annulation),
        len(X_depart_4_mois),
        len(X_depart_6_mois),
        len(X_depart_12_mois),
        len(X_impaye)
    ],
    "Lignes dans y": [
        len(y_annulation),
        len(y_depart_4_mois),
        len(y_depart_6_mois),
        len(y_depart_12_mois),
        len(y_impaye)
    ],
    "Cas positifs": [
        y_annulation.sum(),
        y_depart_4_mois.sum(),
        y_depart_6_mois.sum(),
        y_depart_12_mois.sum(),
        y_impaye.sum()
    ]
})

controle_modelisation

,Sujet,Lignes dans X,Lignes dans y,Cas positifs
0,Annulation,9861,9861,1295
1,Départ 4 mois,7914,7914,2850
2,Départ 6 mois,7737,7737,3506
3,Départ 12 mois,7271,7271,4659
4,Impayé,8566,8566,69


## 7. Modélisation de l’annulation avant location

### 7.1 Séparation temporelle des données

Le modèle est évalué sur une période postérieure à celle utilisée pour son entraînement :

- les contrats antérieurs ou égaux à 2024 servent à l’apprentissage ;
- les contrats de 2025 servent au test ;
- les contrats de 2026 sont réservés à l’application future du modèle définitif.

Cette séparation temporelle est plus proche d’une utilisation opérationnelle qu’une répartition aléatoire des contrats.

In [51]:
annee_annulation = regroupement_centre.loc[
    population_annulation.index,
    "Rent starts on"
].dt.year

In [52]:
controle_annulation_par_annee = pd.DataFrame({
    "Année": annee_annulation,
    "Cible_annulation": y_annulation
})

controle_annulation_par_annee = (
    controle_annulation_par_annee
    .groupby("Année")
    .agg(
        Nombre_contrats=("Cible_annulation", "size"),
        Nombre_annulations=("Cible_annulation", "sum")
    )
)

controle_annulation_par_annee["Taux_annulation (%)"] = (
    controle_annulation_par_annee["Nombre_annulations"]
    / controle_annulation_par_annee["Nombre_contrats"]
    * 100
).round(2)

controle_annulation_par_annee

,Nombre_contrats,Nombre_annulations,Taux_annulation (%)
Année,,,
1997,1,0,0.00
1998,1,0,0.00
1999,3,0,0.00
2001,3,0,0.00
2002,4,0,0.00
2003,3,0,0.00
2004,2,0,0.00
2005,8,0,0.00
2006,3,0,0.00


In [53]:
masque_train_annulation = (
    (annee_annulation >= 2022)
    & (annee_annulation <= 2024)
)

masque_test_annulation = (
    annee_annulation == 2025
)

masque_2026_annulation = (
    annee_annulation == 2026
)

In [54]:
X_train_annulation = X_annulation.loc[
    masque_train_annulation
].copy()

y_train_annulation = y_annulation.loc[
    masque_train_annulation
].copy()

X_test_annulation = X_annulation.loc[
    masque_test_annulation
].copy()

y_test_annulation = y_annulation.loc[
    masque_test_annulation
].copy()

X_2026_annulation = X_annulation.loc[
    masque_2026_annulation
].copy()

In [55]:
controle_decoupage_annulation = pd.Series({
    "Entraînement 2022-2024": len(X_train_annulation),
    "Annulations entraînement": y_train_annulation.sum(),
    "Taux entraînement (%)": y_train_annulation.mean() * 100,

    "Test 2025": len(X_test_annulation),
    "Annulations test": y_test_annulation.sum(),
    "Taux test (%)": y_test_annulation.mean() * 100,

    "Contrats 2026 réservés": len(X_2026_annulation)
}).round(2)

controle_decoupage_annulation

Entraînement 2022-2024      5806.00
Annulations entraînement     646.00
Taux entraînement (%)         11.13
Test 2025                   2270.00
Annulations test             346.00
Taux test (%)                 15.24
Contrats 2026 réservés      1393.00
dtype: float64

Les contrats antérieurs à 2022 sont exclus de la modélisation, car aucune annulation n’y est enregistrée. Cette absence semble davantage refléter une limite de l’historique disponible qu’une absence réelle d’annulations. Leur intégration risquerait donc de créer un biais temporel.

### 7.2 Prétraitement des variables

Les variables catégorielles doivent être transformées en colonnes numériques avant l’entraînement. Les variables numériques sont standardisées pour la régression logistique.

Le prétraitement est intégré dans une pipeline afin qu’il soit appris uniquement sur les données d’entraînement. Cette méthode évite d’utiliser indirectement des informations provenant de l’année de test.

Le déséquilibre de la cible est pris en compte avec l’option `class_weight="balanced

In [56]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [57]:
preprocesseur = ColumnTransformer(
    transformers=[
        (
            "variables_categorielles",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            variables_categorielles
        ),
        (
            "variables_numeriques",
            StandardScaler(),
            variables_numeriques
        )
    ]
)

preprocesseur

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('variables_categorielles', ...), ('variables_numeriques', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transforme

### 7.3 Modèles comparés

Trois modèles de classification sont comparés :

- la régression logistique, simple et interprétable ;
- l’arbre de décision, capable de représenter des règles non linéaires ;
- la forêt aléatoire, plus robuste grâce à la combinaison de plusieurs arbres.

Les mêmes variables et la même période de test sont utilisées pour garantir une comparaison équitable.

In [58]:
modele_logistique_annulation = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

modele_arbre_annulation = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=5,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

modele_foret_annulation = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

### 7.4 Entraînement et comparaison des performances

Les trois modèles sont entraînés sur les contrats de 2022 à 2024, puis évalués sur les contrats de 2025.

Le seuil de classification est fixé à `0,50` pour cette première comparaison. Plusieurs indicateurs sont calculés :

- `Recall` : proportion des annulations effectivement détectées ;
- `Precision` : fiabilité des alertes émises ;
- `F1-score` : équilibre entre le rappel et la précision ;
- `ROC-AUC` : capacité générale à classer les contrats ;
- `Average Precision` : performance du classement en tenant compte de la rareté des annulations.

L’accuracy est présentée à titre indicatif, mais elle ne suffit pas pour choisir le meilleur modèle.

In [59]:
modeles_annulation = {
    "Régression logistique":
        modele_logistique_annulation,
    "Arbre de décision":
        modele_arbre_annulation,
    "Random Forest":
        modele_foret_annulation
}

resultats_annulation = []

for nom_modele, modele in modeles_annulation.items():

    # Entraînement
    modele.fit(
        X_train_annulation,
        y_train_annulation
    )

    # Probabilité d’annulation
    probabilites = modele.predict_proba(
        X_test_annulation
    )[:, 1]

    # Classification avec le seuil de 0,50
    predictions = (
        probabilites >= 0.50
    ).astype(int)

    # Matrice de confusion
    tn, fp, fn, tp = confusion_matrix(
        y_test_annulation,
        predictions
    ).ravel()

    resultats_annulation.append({
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(
            y_test_annulation,
            predictions
        ),
        "Precision": precision_score(
            y_test_annulation,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_annulation,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test_annulation,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_annulation,
            probabilites
        ),
        "Average Precision":
            average_precision_score(
                y_test_annulation,
                probabilites
            ),
        "Vrais positifs": tp,
        "Faux positifs": fp,
        "Faux négatifs": fn,
        "Vrais négatifs": tn })

resultats_annulation = pd.DataFrame(
    resultats_annulation
)

colonnes_metriques = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "Average Precision"
]

resultats_annulation[
    colonnes_metriques
] = resultats_annulation[
    colonnes_metriques
].round(3)

resultats_annulation

,Modèle,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision,Vrais positifs,Faux positifs,Faux négatifs,Vrais négatifs
0,Régression logistique,0.450,0.167,0.656,0.267,0.549,0.183,227,1130,119,794
1,Arbre de décision,0.620,0.224,0.607,0.327,0.685,0.229,210,727,136,1197
2,Random Forest,0.701,0.226,0.396,0.288,0.694,0.231,137,469,209,1455


### 7.5 Interprétation des résultats

L’évaluation temporelle sur les contrats de 2025 montre que les variables disponibles permettent de distinguer partiellement les contrats annulés, mais qu’elles ne suffisent pas à produire une prédiction individuelle très fiable.

La régression logistique détecte le plus grand nombre d’annulations, mais génère 1 130 fausses alertes. Son utilisation opérationnelle entraînerait donc un volume de vérifications trop important.

Le Random Forest présente la meilleure ROC-AUC (`0,694`) et la meilleure Average Precision (`0,231`), mais il ne détecte que 39,6 % des annulations avec le seuil de 0,50.

Avec ce même seuil, l’arbre de décision offre le meilleur compromis entre détection et volume d’alertes. Il identifie 210 des 346 annulations de 2025, mais génère encore 727 fausses alertes. Sa précision de 22,4 % signifie qu’environ une alerte sur quatre correspond effectivement à une annulation.

L’arbre de décision est donc retenu provisoirement pour poursuivre l’analyse des profils. Le modèle ne doit toutefois pas déclencher de décision automatique : il peut uniquement servir à prioriser des actions légères de confirmation ou de suivi commercial.

Les performances modestes suggèrent que des informations supplémentaires seraient nécessaires, notamment la date de signature, le délai entre la réservation et le démarrage, le canal d’acquisition, le motif de location, les échanges commerciaux et le motif d’annulation.

### 7.6 Identification du profil associé aux annulations

Cette étape cherche à identifier les caractéristiques les plus fréquemment associées aux annulations avant location.

L’analyse repose sur deux approches complémentaires :

1. mesurer l’importance des variables dans le modèle ;
2. comparer les taux d’annulation réellement observés selon les différentes caractéristiques des contrats.

Une variable importante pour le modèle n’est pas nécessairement la cause de l’annulation. Elle peut également représenter une offre commerciale, une période ou une catégorie de box particulière.

In [60]:
importance_annulation = permutation_importance(
    modele_arbre_annulation,
    X_test_annulation,
    y_test_annulation,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importance_variables_annulation = pd.DataFrame({
    "Variable": X_test_annulation.columns,
    "Importance moyenne":
        importance_annulation.importances_mean,
    "Écart-type":
        importance_annulation.importances_std
})

importance_variables_annulation = (
    importance_variables_annulation
    .sort_values(
        by="Importance moyenne",
        ascending=False
    )
    .reset_index(drop=True)
)

importance_variables_annulation[
    ["Importance moyenne", "Écart-type"]
] = importance_variables_annulation[
    ["Importance moyenne", "Écart-type"]
].round(4)

importance_variables_annulation

,Variable,Importance moyenne,Écart-type
0,Insurance Premium,0.0795,0.0038
1,Area,0.0161,0.0056
2,Discount (%),0.0077,0.0032
3,Discount duration,0.0012,0.0017
4,Payment mode,0.0004,0.0002
5,Centre,0.0002,0.0001
6,Tax User Type,0.0000,0.0000
7,A_service,0.0000,0.0000
8,Recurring Period,0.0000,0.0000
9,Prix_standard_m2,-0.0034,0.0015


Ces résultats montrent qu’un profil commence à se dégager, mais l’importance ne donne pas encore le sens de la relation.
Insurance Premium est nettement la variable la plus utilisée par le modèle.
Area arrive loin derrière.
Le taux et la durée de remise apportent un peu d’information.
Les autres variables ont très peu d’influence dans ce modèle.
Une importance négative signifie que la variable n’améliore pas réellement les prédictions sur le test.
Attention : cela ne signifie pas que l’assurance provoque l’annulation. La prime dépend notamment de la surface et de la valeur des biens assurés. Elle peut donc représenter indirectement un type de contrat ou de client.

### 7.7 Analyse concrète du profil d’annulation
Nous allons maintenant regarder dans les contrats de test 2025 quels niveaux de prime, surfaces et remises présentent effectivement davantage d’annulations.

In [61]:
# Reconstituer les données du test 2025 avec la cible réelle
profil_annulation_2025 = X_test_annulation.copy()

profil_annulation_2025["Annulation"] = (
    y_test_annulation
    .reindex(profil_annulation_2025.index)
    .astype(int)
)

# Création de tranches de surface
profil_annulation_2025["Tranche_surface"] = pd.cut(
    profil_annulation_2025["Area"],
    bins=[0, 2, 3, 5, 8, float("inf")],
    labels=[
        "Moins de 2 m²",
        "De 2 à moins de 3 m²",
        "De 3 à moins de 5 m²",
        "De 5 à moins de 8 m²",
        "8 m² et plus"
    ],
    include_lowest=True,
    right=False
)

# Création de tranches de remise
profil_annulation_2025["Tranche_remise"] = pd.cut(
    profil_annulation_2025["Discount (%)"],
    bins=[-0.01, 0, 20, 40, 60, 100],
    labels=[
        "Aucune remise",
        "Plus de 0 à 20 %",
        "Plus de 20 à 40 %",
        "Plus de 40 à 60 %",
        "Plus de 60 %"
    ],
    include_lowest=True
)

def analyser_taux_annulation(data, variable):
    resultat = (
        data.groupby(variable, observed=True)
        .agg(
            Nombre_contrats=("Annulation", "size"),
            Nombre_annulations=("Annulation", "sum")
        )
    )

    resultat["Taux_annulation (%)"] = (
        resultat["Nombre_annulations"]
        / resultat["Nombre_contrats"]
        * 100
    ).round(2)

    return resultat.sort_values(
        "Taux_annulation (%)",
        ascending=False
    )

print("=== ANNULATION PAR PRIME D'ASSURANCE ===")
display(
    analyser_taux_annulation(
        profil_annulation_2025,
        "Insurance Premium"
    )
)

print("=== ANNULATION PAR TRANCHE DE SURFACE ===")
display(
    analyser_taux_annulation(
        profil_annulation_2025,
        "Tranche_surface"
    )
)

print("=== ANNULATION PAR TRANCHE DE REMISE ===")
display(
    analyser_taux_annulation(
        profil_annulation_2025,
        "Tranche_remise"
    )
)

print("=== ANNULATION PAR DURÉE DE REMISE ===")
display(
    analyser_taux_annulation(
        profil_annulation_2025,
        "Discount duration"
    )
)

=== ANNULATION PAR PRIME D'ASSURANCE ===


,Nombre_contrats,Nombre_annulations,Taux_annulation (%)
Insurance Premium,,,
2.00,1,1,100.00
50.00,1,1,100.00
5.90,3,3,100.00
5.00,3,2,66.67
12.90,66,23,34.85
7.00,12,4,33.33
15.90,109,34,31.19
34.90,10,3,30.00
39.90,4,1,25.00


=== ANNULATION PAR TRANCHE DE SURFACE ===


,Nombre_contrats,Nombre_annulations,Taux_annulation (%)
Tranche_surface,,,
Moins de 2 m²,425,93,21.88
De 3 à moins de 5 m²,540,83,15.37
De 2 à moins de 3 m²,326,46,14.11
De 5 à moins de 8 m²,513,67,13.06
8 m² et plus,466,57,12.23


=== ANNULATION PAR TRANCHE DE REMISE ===


,Nombre_contrats,Nombre_annulations,Taux_annulation (%)
Tranche_remise,,,
Plus de 60 %,248,46,18.55
Plus de 40 à 60 %,234,43,18.38
Plus de 20 à 40 %,800,131,16.38
Plus de 0 à 20 %,258,39,15.12
Aucune remise,730,87,11.92


=== ANNULATION PAR DURÉE DE REMISE ===


,Nombre_contrats,Nombre_annulations,Taux_annulation (%)
Discount duration,,,
239.0,1,1,100.00
240.0,5,3,60.00
4.0,6,2,33.33
1.0,34,9,26.47
12.0,763,137,17.96
2.0,436,76,17.43
11.0,70,10,14.29
5.0,8,1,12.50
0.0,727,87,11.97


Profil le plus susceptible d’annuler avant le début de la location
D’après les contrats observés en 2025, le profil le plus associé à l’annulation est :

— Un client réservant un petit box de moins de 2 m², 

— bénéficiant d’une remise commerciale importante notamment supérieure à 40 %,  

— et associé à certaines primes d’assurance, notamment 9,90 €, 12,90 € ou 15,90 €.

Les deux caractéristiques les plus lisibles sont :

Une surface inférieure à 2 m² : 21,88 % d’annulations ;

Une remise supérieure à 40 % : environ 18,5 % d’annulations, contre 11,92 % sans remise.

La prime d’assurance est la variable la plus importante pour le modèle, mais elle dépend des caractéristiques du contrat et des biens assurés. Elle doit donc être considérée comme un signal complémentaire, et non comme une cause d’annulation.

Les annulations avant location concernent davantage les petits box et les contrats fortement remisés. Ce profil peut servir à cibler une confirmation renforcée du besoin avant le démarrage, sans automatiser le refus ou l’annulation d’un contrat.

Il s’agit d’un profil statistiquement associé aux annulations, pas d’une certitude individuelle ni d’une relation de cause à effet.

## 8.1 Modélisation du départ dans les 4 mois

L’objectif est d’identifier les caractéristiques connues au début du contrat qui sont associées à un départ pendant les quatre premiers mois de location.

Ce modèle constitue un outil d’aide à la fidélisation. Il ne doit pas être utilisé pour prendre automatiquement une décision concernant un client.

In [62]:
controle_depart_4_mois_par_annee = (
    population_depart_4_mois
    .assign(
        Annee_debut=population_depart_4_mois[
            "Rent starts on"
        ].dt.year
    )
    .groupby("Annee_debut")
    .agg(
        Nombre_contrats=(
            "Cible_depart_4_mois",
            "size"
        ),
        Nombre_departs=(
            "Cible_depart_4_mois",
            "sum"
        )
    )
)

controle_depart_4_mois_par_annee[
    "Taux_depart (%)"
] = (
    controle_depart_4_mois_par_annee[
        "Nombre_departs"
    ]
    / controle_depart_4_mois_par_annee[
        "Nombre_contrats"
    ]
    * 100
).round(2)

display(controle_depart_4_mois_par_annee)

,Nombre_contrats,Nombre_departs,Taux_depart (%)
Annee_debut,,,
1997,1,0,0.00
1998,1,0,0.00
1999,3,0,0.00
2001,3,0,0.00
2002,4,0,0.00
2003,3,0,0.00
2004,2,0,0.00
2005,8,0,0.00
2006,3,0,0.00


### 8.2 Séparation temporelle des données

Le modèle est entraîné sur les contrats commencés entre 2022 et 2024, puis testé sur ceux de 2025.

Les contrats de 2026 sont réservés pour une application ultérieure du modèle. Les années antérieures à 2022 sont écartées en raison de leurs faibles volumes et de l’absence de départs identifiés dans les quatre premiers mois.

In [63]:
# Préparation de la population du départ à 4 mois
donnees_depart_4_mois = donnees_modelisation.loc[
    population_depart_4_mois.index
].copy()

donnees_depart_4_mois["Cible_depart_4_mois"] = (
    population_depart_4_mois["Cible_depart_4_mois"]
    .astype(int)
)

donnees_depart_4_mois["Annee_debut"] = (
    population_depart_4_mois["Rent starts on"].dt.year
)

# Variables explicatives déjà définies dans la section précédente
X_depart_4_mois = donnees_depart_4_mois[
    variables_categorielles + variables_numeriques
].copy()

y_depart_4_mois = donnees_depart_4_mois[
    "Cible_depart_4_mois"
].copy()

# Masques temporels
masque_train_depart_4 = (
    donnees_depart_4_mois["Annee_debut"]
    .between(2022, 2024)
)

masque_test_depart_4 = (
    donnees_depart_4_mois["Annee_debut"] == 2025
)

masque_2026_depart_4 = (
    donnees_depart_4_mois["Annee_debut"] == 2026
)

# Jeux d'entraînement et de test
X_train_depart_4 = X_depart_4_mois.loc[
    masque_train_depart_4
].copy()

y_train_depart_4 = y_depart_4_mois.loc[
    masque_train_depart_4
].copy()

X_test_depart_4 = X_depart_4_mois.loc[
    masque_test_depart_4
].copy()

y_test_depart_4 = y_depart_4_mois.loc[
    masque_test_depart_4
].copy()

# Population 2026 conservée à part
X_2026_depart_4 = X_depart_4_mois.loc[
    masque_2026_depart_4
].copy()

print("Entraînement 2022-2024 :", len(X_train_depart_4))
print("Départs entraînement :", y_train_depart_4.sum())
print(
    "Taux entraînement (%) :",
    round(y_train_depart_4.mean() * 100, 2)
)

print("\nTest 2025 :", len(X_test_depart_4))
print("Départs test :", y_test_depart_4.sum())
print(
    "Taux test (%) :",
    round(y_test_depart_4.mean() * 100, 2)
)

print("\nContrats 2026 réservés :", len(X_2026_depart_4))

Entraînement 2022-2024 : 5094
Départs entraînement : 1907
Taux entraînement (%) : 37.44

Test 2025 : 1921
Départs test : 674
Taux test (%) : 35.09

Contrats 2026 réservés : 507


### 8.3 Comparaison des modèles

Trois algorithmes sont comparés :

- la régression logistique, simple à interpréter ;
- l’arbre de décision, capable d’identifier des profils compréhensibles ;
- la Random Forest, généralement plus performante mais moins directement interprétable.

Les performances sont évaluées sur les contrats commencés en 2025, jamais utilisés pendant l’entraînement.

In [64]:
# Modèles du départ dans les 4 mois
modele_logistique_depart_4 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

modele_arbre_depart_4 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=5,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

modele_foret_depart_4 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

modeles_depart_4 = {
    "Régression logistique": modele_logistique_depart_4,
    "Arbre de décision": modele_arbre_depart_4,
    "Random Forest": modele_foret_depart_4
}

resultats_depart_4 = []

for nom_modele, modele in modeles_depart_4.items():

    # Entraînement
    modele.fit(
        X_train_depart_4,
        y_train_depart_4
    )

    # Prédictions sur 2025
    predictions = modele.predict(
        X_test_depart_4
    )

    probabilites = modele.predict_proba(
        X_test_depart_4
    )[:, 1]

    # Matrice de confusion
    tn, fp, fn, tp = confusion_matrix(
        y_test_depart_4,
        predictions
    ).ravel()

    resultats_depart_4.append({
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(
            y_test_depart_4,
            predictions
        ),
        "Precision": precision_score(
            y_test_depart_4,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_depart_4,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test_depart_4,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_depart_4,
            probabilites
        ),
        "Average Precision": average_precision_score(
            y_test_depart_4,
            probabilites
        ),
        "Vrais positifs": tp,
        "Faux positifs": fp,
        "Faux négatifs": fn,
        "Vrais négatifs": tn
    })

comparaison_depart_4 = pd.DataFrame(
    resultats_depart_4
)

colonnes_metriques = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "Average Precision"
]

comparaison_depart_4[
    colonnes_metriques
] = comparaison_depart_4[
    colonnes_metriques
].round(3)

display(comparaison_depart_4)

,Modèle,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision,Vrais positifs,Faux positifs,Faux négatifs,Vrais négatifs
0,Régression logistique,0.573,0.438,0.761,0.556,0.665,0.523,513,659,161,588
1,Arbre de décision,0.755,0.636,0.706,0.669,0.825,0.662,476,272,198,975
2,Random Forest,0.790,0.716,0.666,0.690,0.857,0.723,449,178,225,1069


### Interprétation des performances

La Random Forest présente les meilleures performances globales sur les contrats de 2025, avec une ROC-AUC de 0,857 et une Average Precision de 0,723.

Au seuil de classification de 50 %, elle identifie correctement 449 des 674 départs observés dans les quatre premiers mois. Parmi les 627 contrats signalés à risque, 449 sont effectivement partis, soit une précision de 71,6 %.

Le modèle constitue donc une base pertinente pour classer les contrats selon leur niveau de vigilance. Il ne détecte cependant pas tous les départs et produit également des fausses alertes. Son score doit être utilisé pour organiser des actions de fidélisation, et non pour prendre automatiquement une décision concernant un client.

### 8.4 Variables associées au départ dans les quatre mois

In [65]:
importance_depart_4 = permutation_importance(
    modele_foret_depart_4,
    X_test_depart_4,
    y_test_depart_4,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importance_variables_depart_4 = pd.DataFrame({
    "Variable": X_test_depart_4.columns,
    "Importance moyenne": importance_depart_4.importances_mean,
    "Écart-type": importance_depart_4.importances_std
})

importance_variables_depart_4 = (
    importance_variables_depart_4
    .sort_values(
        "Importance moyenne",
        ascending=False
    )
    .reset_index(drop=True)
)

importance_variables_depart_4[
    ["Importance moyenne", "Écart-type"]
] = importance_variables_depart_4[
    ["Importance moyenne", "Écart-type"]
].round(4)

display(importance_variables_depart_4)

,Variable,Importance moyenne,Écart-type
0,Insurance Premium,0.2344,0.0077
1,Discount duration,0.0561,0.0120
2,Discount (%),0.0315,0.0109
3,A_service,0.0224,0.0057
4,Tax User Type,0.0069,0.0030
5,Mois_debut,0.0069,0.0062
6,Payment mode,0.0023,0.0020
7,Area,0.0021,0.0038
8,Centre,0.0011,0.0045
9,Prix_standard_m2,0.0001,0.0041


### 8.5 Profil associé au départ dans les quatre mois

L’importance des variables indique les informations utilisées par le modèle, mais pas le sens de leur relation avec le départ.

Les taux réellement observés en 2025 sont donc analysés par prime d’assurance, niveau de remise, durée de remise, souscription d’un service, type de client et mois de début.

Les résultats doivent toujours être interprétés en tenant compte du nombre de contrats dans chaque catégorie.

In [66]:
# Reconstitution des contrats du test 2025
profil_depart_4_2025 = X_test_depart_4.copy()

profil_depart_4_2025["Depart_4_mois"] = (
    y_test_depart_4
    .reindex(profil_depart_4_2025.index)
    .astype(int)
)

# Libellé plus lisible pour les services
profil_depart_4_2025["Service_souscrit"] = (
    profil_depart_4_2025["A_service"]
    .map({
        0: "Aucun service",
        1: "Service souscrit"
    })
)

# Tranches de remise
profil_depart_4_2025["Tranche_remise"] = pd.cut(
    profil_depart_4_2025["Discount (%)"],
    bins=[-0.01, 0, 20, 40, 60, 100],
    labels=[
        "Aucune remise",
        "Plus de 0 à 20 %",
        "Plus de 20 à 40 %",
        "Plus de 40 à 60 %",
        "Plus de 60 %"
    ],
    include_lowest=True
)

# Noms des mois
noms_mois = {
    1: "Janvier",
    2: "Février",
    3: "Mars",
    4: "Avril",
    5: "Mai",
    6: "Juin",
    7: "Juillet",
    8: "Août",
    9: "Septembre",
    10: "Octobre",
    11: "Novembre",
    12: "Décembre"
}

profil_depart_4_2025["Nom_mois_debut"] = (
    profil_depart_4_2025["Mois_debut"]
    .map(noms_mois)
)

def analyser_taux_depart_4(data, variable):
    resultat = (
        data.groupby(variable, observed=True, dropna=False)
        .agg(
            Nombre_contrats=("Depart_4_mois", "size"),
            Nombre_departs=("Depart_4_mois", "sum")
        )
    )

    resultat["Taux_depart_4_mois (%)"] = (
        resultat["Nombre_departs"]
        / resultat["Nombre_contrats"]
        * 100
    ).round(2)

    return resultat.sort_values(
        "Taux_depart_4_mois (%)",
        ascending=False
    )

print("=== DÉPART PAR PRIME D’ASSURANCE ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Insurance Premium"
    )
)

print("=== DÉPART PAR TRANCHE DE REMISE ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Tranche_remise"
    )
)

print("=== DÉPART PAR DURÉE DE REMISE ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Discount duration"
    )
)

print("=== DÉPART SELON LA SOUSCRIPTION D’UN SERVICE ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Service_souscrit"
    )
)

print("=== DÉPART PAR TYPE DE CLIENT ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Tax User Type"
    )
)

print("=== DÉPART PAR MOIS DE DÉBUT ===")
display(
    analyser_taux_depart_4(
        profil_depart_4_2025,
        "Nom_mois_debut"
    )
)

=== DÉPART PAR PRIME D’ASSURANCE ===


,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
Insurance Premium,,,
5.00,1,1,100.00
35.00,1,1,100.00
65.90,1,1,100.00
54.90,5,5,100.00
39.90,3,3,100.00
7.00,8,7,87.50
34.90,7,6,85.71
9.00,5,4,80.00
75.90,8,6,75.00


=== DÉPART PAR TRANCHE DE REMISE ===


,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
Tranche_remise,,,
Plus de 60 %,202,130,64.36
Plus de 40 à 60 %,191,119,62.30
Aucune remise,643,207,32.19
Plus de 0 à 20 %,217,61,28.11
Plus de 20 à 40 %,668,157,23.50


=== DÉPART PAR DURÉE DE REMISE ===


...

,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
Service_souscrit,,,
Service souscrit,940,429,45.64
Aucun service,981,245,24.97


=== DÉPART PAR TYPE DE CLIENT ===


,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
Tax User Type,,,
Student,11,6,54.55
General,1710,608,35.56
Business,199,60,30.15
Charitable,1,0,0.00


=== DÉPART PAR MOIS DE DÉBUT ===


,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
Nom_mois_debut,,,
Décembre,90,44,48.89
Janvier,146,71,48.63
Août,205,86,41.95
Juin,157,65,41.40
Mai,161,66,40.99
Septembre,126,46,36.51
Avril,143,51,35.66
Juillet,206,72,34.95
Février,123,39,31.71


### Profil associé au départ dans les quatre mois

L’analyse des contrats commencés en 2025 fait apparaître un profil de vigilance principalement associé aux conditions commerciales du contrat.

Les contrats bénéficiant d’une remise supérieure à 40 % présentent un taux de départ proche de 63 à 64 %, contre 35,09 % pour l’ensemble de la population. Le risque est également plus élevé lorsque la remise est enregistrée sur une durée de un ou deux mois.

La souscription d’un service est associée à 45,64 % de départs précoces, contre 24,97 % en l’absence de service. Ce résultat peut traduire l’existence de formules ou de besoins spécifiques et ne signifie pas que le service provoque le départ.

Une saisonnalité apparaît également : les contrats commencés en décembre et janvier présentent près de 49 % de départs sous quatre mois, tandis que ceux commencés en octobre n’enregistrent que 16,39 %.

Le profil nécessitant une vigilance renforcée correspond donc plutôt à une location assortie d’une remise forte et courte, éventuellement accompagnée d’un service et commencée pendant une période présentant historiquement davantage de départs.

Ces résultats décrivent des associations statistiques et non des relations de cause à effet. Ils doivent servir à cibler des actions de fidélisation adaptées, sans entraîner de décision automatique concernant le client.

### 8.6 Croisement des principaux facteurs

Les analyses précédentes étudient chaque caractéristique séparément. Cette étape vérifie si le cumul d’une remise élevée, d’une courte durée promotionnelle et d’un service souscrit est associé à un taux de départ encore plus important.

Cette analyse descriptive permet de préciser le profil métier. Elle ne remplace pas le score calculé par le modèle.

In [67]:
# Création des principaux facteurs de vigilance
profil_depart_4_2025["Remise_superieure_40"] = (
    profil_depart_4_2025["Discount (%)"] > 40
)

profil_depart_4_2025["Remise_courte"] = (
    profil_depart_4_2025["Discount duration"].isin([1, 2])
)

profil_depart_4_2025["Avec_service"] = (
    profil_depart_4_2025["A_service"] == 1
)

# Nombre de facteurs cumulés par contrat
profil_depart_4_2025["Nombre_facteurs"] = (
    profil_depart_4_2025[
        [
            "Remise_superieure_40",
            "Remise_courte",
            "Avec_service"
        ]
    ]
    .astype(int)
    .sum(axis=1)
)

# Libellé du cumul
profil_depart_4_2025["Profil_facteurs"] = (
    profil_depart_4_2025["Nombre_facteurs"]
    .map({
        0: "Aucun facteur",
        1: "Un facteur",
        2: "Deux facteurs",
        3: "Trois facteurs"
    })
)

# Résultats observés selon le nombre de facteurs
analyse_cumul_facteurs = (
    profil_depart_4_2025
    .groupby(
        ["Nombre_facteurs", "Profil_facteurs"],
        observed=True
    )
    .agg(
        Nombre_contrats=("Depart_4_mois", "size"),
        Nombre_departs=("Depart_4_mois", "sum")
    )
    .reset_index()
)

analyse_cumul_facteurs["Taux_depart_4_mois (%)"] = (
    analyse_cumul_facteurs["Nombre_departs"]
    / analyse_cumul_facteurs["Nombre_contrats"]
    * 100
).round(2)

analyse_cumul_facteurs = analyse_cumul_facteurs.sort_values(
    "Nombre_facteurs"
)

display(analyse_cumul_facteurs)

,Nombre_facteurs,Profil_facteurs,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
0,0,Aucun facteur,843,167,19.81
1,1,Un facteur,694,258,37.18
2,2,Deux facteurs,128,80,62.50
3,3,Trois facteurs,256,169,66.02


In [68]:
analyse_combinaisons = (
    profil_depart_4_2025
    .groupby(
        [
            "Remise_superieure_40",
            "Remise_courte",
            "Avec_service"
        ],
        observed=True
    )
    .agg(
        Nombre_contrats=("Depart_4_mois", "size"),
        Nombre_departs=("Depart_4_mois", "sum")
    )
    .reset_index()
)

analyse_combinaisons["Taux_depart_4_mois (%)"] = (
    analyse_combinaisons["Nombre_departs"]
    / analyse_combinaisons["Nombre_contrats"]
    * 100
).round(2)

analyse_combinaisons = analyse_combinaisons.sort_values(
    ["Taux_depart_4_mois (%)", "Nombre_contrats"],
    ascending=[False, False]
)

display(analyse_combinaisons)

,Remise_superieure_40,Remise_courte,Avec_service,Nombre_contrats,Nombre_departs,Taux_depart_4_mois (%)
7,True,True,True,256,169,66.02
6,True,True,False,120,77,64.17
5,True,False,True,7,3,42.86
1,False,False,True,676,257,38.02
0,False,False,False,843,167,19.81
2,False,True,False,8,1,12.50
4,True,False,False,10,0,0.00
3,False,True,True,1,0,0.00


### Conclusion sur le profil de départ dans les quatre mois

Le croisement des principaux facteurs confirme l’existence d’un profil de vigilance identifiable.

Parmi les contrats ne présentant aucun des trois facteurs étudiés 

— remise supérieure à 40 %, 

— remise de un ou deux mois 

—service souscrit — 19,81 % se terminent dans les quatre premiers mois.

Ce taux atteint 37,18 % lorsqu’un facteur est présent, 62,50 % avec deux facteurs et 66,02 % lorsque les trois facteurs sont réunis.

L’analyse détaillée montre cependant que le signal principal repose sur la combinaison d’une remise supérieure à 40 % et d’une durée promotionnelle de un ou deux mois. Le taux de départ est de 64,17 % sans service et de 66,02 % avec un service. La souscription d’un service n’augmente donc que faiblement le risque lorsque les deux caractéristiques promotionnelles sont déjà réunies.

Ces résultats suggèrent que les promotions fortes et courtes sont fréquemment associées à des locations répondant à un besoin temporaire. Elles ne doivent pas conduire à refuser le contrat, mais peuvent déclencher un accompagnement commercial adapté : qualification du besoin, estimation de la durée envisagée et prise de contact avant la fin de la période promotionnelle.

Cette relation reste une association statistique observée sur les contrats de 2025 et ne démontre pas que la remise provoque le départ.

## 9. Modélisation du départ dans les 6 mois

L’objectif est d’identifier les contrats susceptibles de se terminer pendant les six premiers mois de location.

Comme pour l’horizon de quatre mois, le modèle est entraîné sur les contrats commencés entre 2022 et 2024, testé sur ceux de 2025, puis appliqué aux contrats pertinents de 2026.

Le score obtenu constitue un indicateur de vigilance destiné à organiser les actions de fidélisation. Il ne doit pas entraîner de décision automatique concernant un client.

### 9.1 Préparation et séparation temporelle

In [69]:
# Préparation de la population exploitable à 6 mois
donnees_depart_6_mois = donnees_modelisation.loc[
    population_depart_6_mois.index
].copy()

donnees_depart_6_mois["Cible_depart_6_mois"] = (
    population_depart_6_mois["Cible_depart_6_mois"]
    .astype(int)
)

donnees_depart_6_mois["Annee_debut"] = (
    population_depart_6_mois["Rent starts on"].dt.year
)

# Variables explicatives
X_depart_6_mois = donnees_depart_6_mois[
    variables_categorielles + variables_numeriques
].copy()

y_depart_6_mois = donnees_depart_6_mois[
    "Cible_depart_6_mois"
].copy()

# Séparation temporelle
masque_train_depart_6 = (
    donnees_depart_6_mois["Annee_debut"]
    .between(2022, 2024)
)

masque_test_depart_6 = (
    donnees_depart_6_mois["Annee_debut"] == 2025
)

masque_2026_depart_6 = (
    donnees_depart_6_mois["Annee_debut"] == 2026
)

X_train_depart_6 = X_depart_6_mois.loc[
    masque_train_depart_6
].copy()

y_train_depart_6 = y_depart_6_mois.loc[
    masque_train_depart_6
].copy()

X_test_depart_6 = X_depart_6_mois.loc[
    masque_test_depart_6
].copy()

y_test_depart_6 = y_depart_6_mois.loc[
    masque_test_depart_6
].copy()

X_2026_depart_6 = X_depart_6_mois.loc[
    masque_2026_depart_6
].copy()

# Contrôles
print("Entraînement 2022-2024 :", len(X_train_depart_6))
print("Départs entraînement :", y_train_depart_6.sum())
print(
    "Taux entraînement (%) :",
    round(y_train_depart_6.mean() * 100, 2)
)

print("\nTest 2025 :", len(X_test_depart_6))
print("Départs test :", y_test_depart_6.sum())
print(
    "Taux test (%) :",
    round(y_test_depart_6.mean() * 100, 2)
)

print("\nContrats 2026 réservés :", len(X_2026_depart_6))

Entraînement 2022-2024 : 5094
Départs entraînement : 2384
Taux entraînement (%) : 46.8

Test 2025 : 1921
Départs test : 840
Taux test (%) : 43.73

Contrats 2026 réservés : 330


### 9.2 Comparaison des modèles

Les trois algorithmes précédents sont entraînés sur les contrats commencés entre 2022 et 2024, puis évalués sur les contrats de 2025.

Les résultats permettront de sélectionner le modèle présentant le meilleur équilibre entre la détection des départs et la fiabilité des alertes.

In [70]:
# Modèles du départ dans les 6 mois
modele_logistique_depart_6 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

modele_arbre_depart_6 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=5,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

modele_foret_depart_6 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

modeles_depart_6 = {
    "Régression logistique": modele_logistique_depart_6,
    "Arbre de décision": modele_arbre_depart_6,
    "Random Forest": modele_foret_depart_6
}

resultats_depart_6 = []

for nom_modele, modele in modeles_depart_6.items():

    modele.fit(
        X_train_depart_6,
        y_train_depart_6
    )

    predictions = modele.predict(
        X_test_depart_6
    )

    probabilites = modele.predict_proba(
        X_test_depart_6
    )[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test_depart_6,
        predictions
    ).ravel()

    resultats_depart_6.append({
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(
            y_test_depart_6,
            predictions
        ),
        "Precision": precision_score(
            y_test_depart_6,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_depart_6,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test_depart_6,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_depart_6,
            probabilites
        ),
        "Average Precision": average_precision_score(
            y_test_depart_6,
            probabilites
        ),
        "Vrais positifs": tp,
        "Faux positifs": fp,
        "Faux négatifs": fn,
        "Vrais négatifs": tn
    })

comparaison_depart_6 = pd.DataFrame(
    resultats_depart_6
)

colonnes_metriques = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "Average Precision"
]

comparaison_depart_6[
    colonnes_metriques
] = comparaison_depart_6[
    colonnes_metriques
].round(3)

display(comparaison_depart_6)

,Modèle,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision,Vrais positifs,Faux positifs,Faux négatifs,Vrais négatifs
0,Régression logistique,0.616,0.543,0.774,0.638,0.680,0.617,650,547,190,534
1,Arbre de décision,0.787,0.826,0.649,0.727,0.878,0.797,545,115,295,966
2,Random Forest,0.811,0.825,0.719,0.768,0.892,0.836,604,128,236,953


### Interprétation des performances à six mois

La Random Forest obtient les meilleures performances sur les contrats de 2025, avec une ROC-AUC de 0,892 et une Average Precision de 0,836.

Elle détecte correctement 604 des 840 départs observés dans les six premiers mois. Parmi les 732 contrats signalés à risque, 604 sont effectivement partis, soit une précision de 82,5 %.

Le modèle à six mois offre donc un bon compromis entre anticipation et fiabilité des alertes. Il reste néanmoins imparfait : 236 départs ne sont pas détectés et 128 contrats sont signalés à tort.

Le score doit servir à prioriser des actions de fidélisation et non à prendre une décision automatique concernant un client.

### 9.3 Importance des variables à six mois

In [71]:
importance_depart_6 = permutation_importance(
    modele_foret_depart_6,
    X_test_depart_6,
    y_test_depart_6,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importance_variables_depart_6 = pd.DataFrame({
    "Variable": X_test_depart_6.columns,
    "Importance moyenne": importance_depart_6.importances_mean,
    "Écart-type": importance_depart_6.importances_std
})

importance_variables_depart_6 = (
    importance_variables_depart_6
    .sort_values(
        "Importance moyenne",
        ascending=False
    )
    .reset_index(drop=True)
)

importance_variables_depart_6[
    ["Importance moyenne", "Écart-type"]
] = importance_variables_depart_6[
    ["Importance moyenne", "Écart-type"]
].round(4)

display(importance_variables_depart_6)

,Variable,Importance moyenne,Écart-type
0,Insurance Premium,0.2656,0.0106
1,Discount duration,0.0389,0.0089
2,A_service,0.0237,0.0062
3,Discount (%),0.0141,0.0075
4,Tax User Type,0.0072,0.0026
5,Mois_debut,0.0036,0.0030
6,Payment mode,0.0018,0.0012
7,Centre,0.0018,0.0046
8,Recurring Period,0.0000,0.0000
9,Prix_standard_m2,-0.0006,0.0029


### Variables associées au départ dans les six mois

Le classement des variables est très proche de celui obtenu pour le départ dans les quatre mois.

La prime d’assurance reste le principal signal utilisé par le modèle. Elle est suivie par la durée de remise, la souscription d’un service et le pourcentage de remise.

Le centre, la surface, le mode de paiement et le prix standard au mètre carré apportent peu d’information supplémentaire.

Cette stabilité renforce l’hypothèse selon laquelle les départs précoces sont principalement associés à certaines formules commerciales, notamment aux offres promotionnelles fortes et courtes. La prime d’assurance doit toutefois être interprétée comme un marqueur indirect du contrat et non comme une cause du départ.

### 9.4 Profils associés aux risques de départ dans les 6 mois 

In [72]:
# Reconstitution des contrats du test 2025
profil_depart_6_2025 = X_test_depart_6.copy()

profil_depart_6_2025["Depart_6_mois"] = (
    y_test_depart_6
    .reindex(profil_depart_6_2025.index)
    .astype(int)
)

# Libellé plus lisible pour les services
profil_depart_6_2025["Service_souscrit"] = (
    profil_depart_6_2025["A_service"]
    .map({
        0: "Aucun service",
        1: "Service souscrit"
    })
)

# Tranches de remise
profil_depart_6_2025["Tranche_remise"] = pd.cut(
    profil_depart_6_2025["Discount (%)"],
    bins=[-0.01, 0, 20, 40, 60, 100],
    labels=[
        "Aucune remise",
        "Plus de 0 à 20 %",
        "Plus de 20 à 40 %",
        "Plus de 40 à 60 %",
        "Plus de 60 %"
    ],
    include_lowest=True
)

# Noms des mois
noms_mois = {
    1: "Janvier",
    2: "Février",
    3: "Mars",
    4: "Avril",
    5: "Mai",
    6: "Juin",
    7: "Juillet",
    8: "Août",
    9: "Septembre",
    10: "Octobre",
    11: "Novembre",
    12: "Décembre"
}

profil_depart_6_2025["Nom_mois_debut"] = (
    profil_depart_6_2025["Mois_debut"]
    .map(noms_mois)
)

def analyser_taux_depart_6(data, variable):
    resultat = (
        data.groupby(variable, observed=True, dropna=False)
        .agg(
            Nombre_contrats=("Depart_6_mois", "size"),
            Nombre_departs=("Depart_6_mois", "sum")
        )
    )

    resultat["Taux_depart_6_mois (%)"] = (
        resultat["Nombre_departs"]
        / resultat["Nombre_contrats"]
        * 100
    ).round(2)

    return resultat.sort_values(
        "Taux_depart_6_mois (%)",
        ascending=False
    )

print("=== DÉPART PAR PRIME D’ASSURANCE ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Insurance Premium"
    )
)

print("=== DÉPART PAR TRANCHE DE REMISE ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Tranche_remise"
    )
)

print("=== DÉPART PAR DURÉE DE REMISE ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Discount duration"
    )
)

print("=== DÉPART SELON LA SOUSCRIPTION D’UN SERVICE ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Service_souscrit"
    )
)

print("=== DÉPART PAR TYPE DE CLIENT ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Tax User Type"
    )
)

print("=== DÉPART PAR MOIS DE DÉBUT ===")
display(
    analyser_taux_depart_6(
        profil_depart_6_2025,
        "Nom_mois_debut"
    )
)

=== DÉPART PAR PRIME D’ASSURANCE ===


,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
Insurance Premium,,,
5.00,1,1,100.00
7.00,8,8,100.00
9.00,5,5,100.00
54.90,5,5,100.00
39.90,3,3,100.00
35.00,1,1,100.00
65.90,1,1,100.00
75.90,8,7,87.50
34.90,7,6,85.71


=== DÉPART PAR TRANCHE DE REMISE ===


,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
Tranche_remise,,,
Plus de 60 %,202,150,74.26
Plus de 40 à 60 %,191,136,71.20
Aucune remise,643,260,40.44
Plus de 0 à 20 %,217,80,36.87
Plus de 20 à 40 %,668,214,32.04


=== DÉPART PAR DURÉE DE REMISE ===


...

,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
Service_souscrit,,,
Service souscrit,940,524,55.74
Aucun service,981,316,32.21


=== DÉPART PAR TYPE DE CLIENT ===


,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
Tax User Type,,,
Student,11,6,54.55
General,1710,766,44.80
Business,199,68,34.17
Charitable,1,0,0.00


=== DÉPART PAR MOIS DE DÉBUT ===


,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
Nom_mois_debut,,,
Décembre,90,50,55.56
Janvier,146,78,53.42
Mai,161,86,53.42
Juin,157,81,51.59
Août,205,105,51.22
Avril,143,67,46.85
Février,123,53,43.09
Septembre,126,54,42.86
Juillet,206,87,42.23


### 9.5 Croissement des facteurs 

In [73]:
# Création des 3 facteurs de risque - DÉPART À 6 MOIS

profil_depart_6_2025["Remise_superieure_40"] = (
    profil_depart_6_2025["Discount (%)"] > 40
)

profil_depart_6_2025["Remise_courte"] = (
    profil_depart_6_2025["Discount duration"].isin([1, 2])
)

profil_depart_6_2025["Avec_service"] = (
    profil_depart_6_2025["A_service"] == 1
)

# Nombre de facteurs présents par contrat
facteurs_6 = [
    "Remise_superieure_40",
    "Remise_courte",
    "Avec_service"
]

profil_depart_6_2025["Nombre_facteurs"] = (
    profil_depart_6_2025[facteurs_6]
    .astype(int)
    .sum(axis=1)
)

# ============================================================
# TABLEAU 1 : TAUX DE DÉPART SELON LE NOMBRE DE FACTEURS
# ============================================================

tableau_facteurs_6 = (
    profil_depart_6_2025
    .groupby("Nombre_facteurs")
    .agg(
        Nombre_contrats=("Depart_6_mois", "size"),
        Nombre_departs=("Depart_6_mois", "sum")
    )
    .reset_index()
)

tableau_facteurs_6["Taux_depart_6_mois (%)"] = (
    tableau_facteurs_6["Nombre_departs"]
    / tableau_facteurs_6["Nombre_contrats"]
    * 100
).round(2)

print("=== TAUX DE DÉPART À 6 MOIS SELON LE NOMBRE DE FACTEURS ===")
display(tableau_facteurs_6)


# ============================================================
# TABLEAU 2 : COMBINAISONS DES 3 FACTEURS
# ============================================================

tableau_combinaisons_6 = (
    profil_depart_6_2025
    .groupby(
        [
            "Remise_superieure_40",
            "Remise_courte",
            "Avec_service"
        ],
        observed=True
    )
    .agg(
        Nombre_contrats=("Depart_6_mois", "size"),
        Nombre_departs=("Depart_6_mois", "sum")
    )
    .reset_index()
)

tableau_combinaisons_6["Taux_depart_6_mois (%)"] = (
    tableau_combinaisons_6["Nombre_departs"]
    / tableau_combinaisons_6["Nombre_contrats"]
    * 100
).round(2)

# Tri du taux le plus élevé au plus faible
tableau_combinaisons_6 = (
    tableau_combinaisons_6
    .sort_values(
        "Taux_depart_6_mois (%)",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=== COMBINAISONS DES FACTEURS - DÉPART À 6 MOIS ===")
display(tableau_combinaisons_6)

=== TAUX DE DÉPART À 6 MOIS SELON LE NOMBRE DE FACTEURS ===


,Nombre_facteurs,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
0,0,843,226,26.81
1,1,694,328,47.26
2,2,128,92,71.88
3,3,256,194,75.78


=== COMBINAISONS DES FACTEURS - DÉPART À 6 MOIS ===


,Remise_superieure_40,Remise_courte,Avec_service,Nombre_contrats,Nombre_departs,Taux_depart_6_mois (%)
0,True,True,True,256,194,75.78
1,True,True,False,120,89,74.17
2,False,False,True,676,327,48.37
3,True,False,True,7,3,42.86
4,False,False,False,843,226,26.81
5,False,True,False,8,1,12.50
6,False,True,True,1,0,0.00
7,True,False,False,10,0,0.00


### Analyse des profils de risque
L'analyse met en évidence une augmentation progressive du risque de départ à mesure que plusieurs facteurs de risque sont réunis.
Les contrats ne présentant aucun facteur identifié affichent un taux de départ de 26,8 %. Ce taux atteint 47,3 % lorsqu'un seul facteur est présent, puis 71,9 % avec deux facteurs et 75,8 % lorsque les trois facteurs sont simultanément réunis.
Cette progression confirme que l'association de plusieurs caractéristiques contractuelles est un meilleur indicateur du risque qu'un facteur isolé. Plus les facteurs de vigilance s'accumulent, plus la probabilité de départ augmente.

### Profil de risque identifié
Le profil présentant le niveau de risque le plus élevé est constitué des contrats cumulant :
- une remise supérieure à 40 % ;
- une remise de courte durée (1 ou 2 mois) ;
- la souscription d'un service complémentaire.
Ces contrats présentent un taux de départ de 75,8 %, soit près de 3 fois supérieur à celui des contrats ne présentant aucun de ces facteurs (26,8 %).

## 10. Modélisation du départ dans les 12 mois

L’objectif est d’identifier les contrats susceptibles de se terminer pendant les douze premiers mois de location.

Ce modèle complète les horizons de quatre et six mois. Il offre une vision plus large du risque de départ, mais nécessite une année complète d’observation pour classer correctement un contrat historique.

Le modèle est entraîné sur les années antérieures, testé sur les contrats de 2025 disposant du recul nécessaire, puis utilisé comme outil de priorisation. Il ne doit pas entraîner de décision automatique concernant un client.

In [74]:
# Date limite garantissant 12 mois complets d'observation
limite_12_mois = date_observation - pd.DateOffset(months=12)

# Sélection des contrats disposant de 12 mois complets d'observation
population_depart_12_mois_corrigee = population_location[
    population_location["Rent starts on"].notna()
    & (population_location["Rent starts on"] <= limite_12_mois)
    & (
        population_location["Duree_location_analyse_jours"].isna()
        | (
            population_location["Duree_location_analyse_jours"]
            >= 0
        )
    )
].copy()

# Création de la cible :
# 1 = départ dans les 12 premiers mois
# 0 = aucun départ dans les 12 premiers mois
population_depart_12_mois_corrigee[
    "Cible_depart_12_mois"
] = (
    population_depart_12_mois_corrigee[
        "Duree_location_analyse_jours"
    ]
    .between(0, 365, inclusive="both")
    .astype(int)
)

# Année de début de la location
population_depart_12_mois_corrigee[
    "Annee_debut"
] = (
    population_depart_12_mois_corrigee[
        "Rent starts on"
    ].dt.year
)

print(
    "Population corrigée à 12 mois :",
    len(population_depart_12_mois_corrigee)
)

print("\nRépartition de la cible :")
print(
    population_depart_12_mois_corrigee[
        "Cible_depart_12_mois"
    ].value_counts()
)

Population corrigée à 12 mois : 6640

Répartition de la cible :
Cible_depart_12_mois
1    3959
0    2681
Name: count, dtype: int64


In [75]:
# Préparation de la population corrigée à 12 mois
donnees_depart_12_mois = donnees_modelisation.loc[
    population_depart_12_mois_corrigee.index
].copy()

# Ajout de la cible
donnees_depart_12_mois["Cible_depart_12_mois"] = (
    population_depart_12_mois_corrigee[
        "Cible_depart_12_mois"
    ].astype(int)
)

# Ajout de l’année de début de location
donnees_depart_12_mois["Annee_debut"] = (
    population_depart_12_mois_corrigee[
        "Rent starts on"
    ].dt.year
)

# Sélection des variables explicatives
X_depart_12_mois = donnees_depart_12_mois[
    variables_categorielles + variables_numeriques
].copy()

# Sélection de la cible
y_depart_12_mois = donnees_depart_12_mois[
    "Cible_depart_12_mois"
].copy()

# Entraînement : contrats commencés entre 2022 et 2024
masque_train_depart_12 = (
    donnees_depart_12_mois["Annee_debut"]
    .between(2022, 2024)
)

# Test : contrats commencés en 2025 et disposant
# déjà de 12 mois complets d’observation
masque_test_depart_12 = (
    donnees_depart_12_mois["Annee_debut"] == 2025
)

# Création du jeu d’entraînement
X_train_depart_12 = X_depart_12_mois.loc[
    masque_train_depart_12
].copy()

y_train_depart_12 = y_depart_12_mois.loc[
    masque_train_depart_12
].copy()

# Création du jeu de test
X_test_depart_12 = X_depart_12_mois.loc[
    masque_test_depart_12
].copy()

y_test_depart_12 = y_depart_12_mois.loc[
    masque_test_depart_12
].copy()

# Vérifications
print("Entraînement 2022-2024 :", len(X_train_depart_12))
print("Départs entraînement :", y_train_depart_12.sum())
print(
    "Taux entraînement (%) :",
    round(y_train_depart_12.mean() * 100, 2)
)

print(
    "\nTest 2025 avec 12 mois complets d’observation :",
    len(X_test_depart_12)
)
print("Départs test :", y_test_depart_12.sum())
print(
    "Taux test (%) :",
    round(y_test_depart_12.mean() * 100, 2)
)

print(
    "\nContrats 2026 évaluables historiquement à 12 mois : 0"
)

Entraînement 2022-2024 : 5160
Départs entraînement : 3250
Taux entraînement (%) : 62.98

Test 2025 avec 12 mois complets d’observation : 1088
Départs test : 709
Taux test (%) : 65.17

Contrats 2026 évaluables historiquement à 12 mois : 0


In [76]:
# Modèles du départ dans les 12 mois

modele_logistique_depart_12 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

modele_arbre_depart_12 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=5,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

modele_foret_depart_12 = Pipeline(
    steps=[
        ("preparation", preprocesseur),
        (
            "modele",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

modeles_depart_12 = {
    "Régression logistique": modele_logistique_depart_12,
    "Arbre de décision": modele_arbre_depart_12,
    "Random Forest": modele_foret_depart_12
}

resultats_depart_12 = []

for nom_modele, modele in modeles_depart_12.items():

    # Entraînement
    modele.fit(
        X_train_depart_12,
        y_train_depart_12
    )

    # Prédictions sur le test 2025
    predictions = modele.predict(
        X_test_depart_12
    )

    probabilites = modele.predict_proba(
        X_test_depart_12
    )[:, 1]

    # Matrice de confusion
    tn, fp, fn, tp = confusion_matrix(
        y_test_depart_12,
        predictions
    ).ravel()

    # Enregistrement des résultats
    resultats_depart_12.append({
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(
            y_test_depart_12,
            predictions
        ),
        "Precision": precision_score(
            y_test_depart_12,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_depart_12,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test_depart_12,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_depart_12,
            probabilites
        ),
        "Average Precision": average_precision_score(
            y_test_depart_12,
            probabilites
        ),
        "Vrais positifs": tp,
        "Faux positifs": fp,
        "Faux négatifs": fn,
        "Vrais négatifs": tn
    })

# Tableau comparatif
comparaison_depart_12 = pd.DataFrame(
    resultats_depart_12
)

colonnes_metriques = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "Average Precision"
]

comparaison_depart_12[
    colonnes_metriques
] = comparaison_depart_12[
    colonnes_metriques
].round(3)

display(comparaison_depart_12)

,Modèle,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision,Vrais positifs,Faux positifs,Faux négatifs,Vrais négatifs
0,Régression logistique,0.683,0.710,0.867,0.781,0.694,0.810,615,251,94,128
1,Arbre de décision,0.691,0.952,0.554,0.701,0.914,0.933,393,20,316,359
2,Random Forest,0.822,0.942,0.774,0.850,0.925,0.958,549,34,160,345


Le Random Forest présente les meilleures performances pour anticiper les départs dans les douze premiers mois. Il identifie 77,4 % des départs observés avec une précision de 94,2 %. Ses résultats sont nettement supérieurs à ceux obtenus aux horizons de quatre et six mois, ce qui indique que les données disponibles permettent davantage d’identifier un profil de location courte que de prévoir précisément un départ imminent. Le modèle reste un outil de priorisation et ne doit pas déclencher automatiquement une action commerciale.

Point important : aucun contrat commencé en 2026 ne possède encore douze mois complets d’observation. On pourra néanmoins appliquer le modèle à ces contrats pour produire un score prévisionnel, mais leur résultat réel ne pourra être contrôlé qu’en 2027.

In [77]:
# Importance des variables du Random Forest à 12 mois

importance_depart_12 = permutation_importance(
    modeles_depart_12["Random Forest"],
    X_test_depart_12,
    y_test_depart_12,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_variables_depart_12 = pd.DataFrame({
    "Variable": X_test_depart_12.columns,
    "Importance moyenne": importance_depart_12.importances_mean,
    "Écart-type": importance_depart_12.importances_std
}).sort_values(
    by="Importance moyenne",
    ascending=False
).reset_index(drop=True)

importance_variables_depart_12[
    ["Importance moyenne", "Écart-type"]
] = importance_variables_depart_12[
    ["Importance moyenne", "Écart-type"]
].round(4)

display(importance_variables_depart_12)

,Variable,Importance moyenne,Écart-type
0,Insurance Premium,0.3229,0.0135
1,Discount duration,0.0229,0.0056
2,Discount (%),0.0064,0.0025
3,A_service,0.0053,0.0026
4,Tax User Type,0.0018,0.0006
5,Prix_standard_m2,0.0004,0.0013
6,Payment mode,0.0001,0.0004
7,Recurring Period,0.0000,0.0000
8,Centre,-0.0009,0.0012
9,Area,-0.0018,0.0012


### Interprétation

La prime d’assurance constitue de très loin la variable la plus utile au modèle pour identifier les départs dans les douze premiers mois. La durée et le pourcentage de remise apportent une information complémentaire, mais leur contribution reste nettement plus faible.

La prime d’assurance ne doit cependant pas être considérée comme une cause directe du départ. Elle dépend notamment du niveau de couverture choisi et peut aussi caractériser certaines périodes tarifaires ou catégories de contrats.

Les résultats permettent donc d’établir un score de vigilance, mais pas d’expliquer à eux seuls le comportement du client. Le modèle devra être contrôlé sur les contrats de 2026 lorsque leur historique sera suffisamment complet.

### 10.1 Profil départ dans les 12 mois 

In [78]:
# ============================================================
# RECONSTITUTION DU JEU DE TEST - DÉPART À 12 MOIS
# ============================================================

profil_depart_12_2025 = X_test_depart_12.copy()

profil_depart_12_2025["Depart_12_mois"] = (
    y_test_depart_12
    .reindex(profil_depart_12_2025.index)
    .astype(int)
)

# Service
profil_depart_12_2025["Service_souscrit"] = (
    profil_depart_12_2025["A_service"]
    .map({
        0: "Aucun service",
        1: "Service souscrit"
    })
)

# Tranches de remise
profil_depart_12_2025["Tranche_remise"] = pd.cut(
    profil_depart_12_2025["Discount (%)"],
    bins=[-0.01,0,20,40,60,100],
    labels=[
        "Aucune remise",
        "Plus de 0 à 20 %",
        "Plus de 20 à 40 %",
        "Plus de 40 à 60 %",
        "Plus de 60 %"
    ],
    include_lowest=True
)

# Analyse générique

def analyser_taux_depart_12(data, variable):

    resultat = (
        data.groupby(variable, observed=True)
        .agg(
            Nombre_contrats=("Depart_12_mois","size"),
            Nombre_departs=("Depart_12_mois","sum")
        )
    )

    resultat["Taux_depart_12_mois (%)"] = (
        resultat["Nombre_departs"]
        / resultat["Nombre_contrats"]
        *100
    ).round(2)

    return resultat.sort_values(
        "Taux_depart_12_mois (%)",
        ascending=False
    )

print("=== PRIME D'ASSURANCE ===")
display(analyser_taux_depart_12(
    profil_depart_12_2025,
    "Insurance Premium"
))

print("=== TRANCHE DE REMISE ===")
display(analyser_taux_depart_12(
    profil_depart_12_2025,
    "Tranche_remise"
))

print("=== DURÉE DE REMISE ===")
display(analyser_taux_depart_12(
    profil_depart_12_2025,
    "Discount duration"
))

=== PRIME D'ASSURANCE ===


,Nombre_contrats,Nombre_departs,Taux_depart_12_mois (%)
Insurance Premium,,,
35.00,1,1,100.00
34.90,6,6,100.00
54.90,4,4,100.00
75.90,3,3,100.00
65.90,1,1,100.00
39.90,2,2,100.00
29.90,24,24,100.00
15.90,60,58,96.67
9.90,553,531,96.02


=== TRANCHE DE REMISE ===


,Nombre_contrats,Nombre_departs,Taux_depart_12_mois (%)
Tranche_remise,,,
Plus de 60 %,202,175,86.63
Plus de 40 à 60 %,115,97,84.35
Aucune remise,94,56,59.57
Plus de 0 à 20 %,145,84,57.93
Plus de 20 à 40 %,532,297,55.83


=== DURÉE DE REMISE ===


,Nombre_contrats,Nombre_departs,Taux_depart_12_mois (%)
Discount duration,,,
3.0,1,1,100.00
2.0,286,251,87.76
1.0,18,15,83.33
0.0,93,55,59.14
11.0,43,25,58.14
12.0,536,303,56.53
120.0,92,51,55.43
5.0,2,1,50.00
119.0,12,6,50.00


In [79]:
# ============================================================
# FACTEURS DE RISQUE - DÉPART À 12 MOIS
# ============================================================

profil_depart_12_2025["Remise_superieure_40"] = (
    profil_depart_12_2025["Discount (%)"] > 40
)

profil_depart_12_2025["Remise_courte"] = (
    profil_depart_12_2025["Discount duration"].isin([1, 2])
)

profil_depart_12_2025["Avec_service"] = (
    profil_depart_12_2025["A_service"] == 1
)

facteurs_12 = [
    "Remise_superieure_40",
    "Remise_courte",
    "Avec_service"
]

profil_depart_12_2025["Nombre_facteurs"] = (
    profil_depart_12_2025[facteurs_12]
    .astype(int)
    .sum(axis=1)
)

# ============================================================
# TABLEAU 1
# ============================================================

tableau_facteurs_12 = (
    profil_depart_12_2025
    .groupby("Nombre_facteurs")
    .agg(
        Nombre_contrats=("Depart_12_mois", "size"),
        Nombre_departs=("Depart_12_mois", "sum")
    )
    .reset_index()
)

tableau_facteurs_12["Taux_depart_12_mois (%)"] = (
    tableau_facteurs_12["Nombre_departs"]
    / tableau_facteurs_12["Nombre_contrats"]
    * 100
).round(2)

display(tableau_facteurs_12)

# ============================================================
# TABLEAU 2
# ============================================================

tableau_combinaisons_12 = (
    profil_depart_12_2025
    .groupby(
        [
            "Remise_superieure_40",
            "Remise_courte",
            "Avec_service"
        ],
        observed=True
    )
    .agg(
        Nombre_contrats=("Depart_12_mois", "size"),
        Nombre_departs=("Depart_12_mois", "sum")
    )
    .reset_index()
)

tableau_combinaisons_12["Taux_depart_12_mois (%)"] = (
    tableau_combinaisons_12["Nombre_departs"]
    / tableau_combinaisons_12["Nombre_contrats"]
    * 100
).round(2)

tableau_combinaisons_12 = (
    tableau_combinaisons_12
    .sort_values(
        "Taux_depart_12_mois (%)",
        ascending=False
    )
)

display(tableau_combinaisons_12)

,Nombre_facteurs,Nombre_contrats,Nombre_departs,Taux_depart_12_mois (%)
0,0,414,195,47.10
1,1,364,243,66.76
2,2,104,94,90.38
3,3,206,177,85.92


,Remise_superieure_40,Remise_courte,Avec_service,Nombre_contrats,Nombre_departs,Taux_depart_12_mois (%)
4,True,True,False,98,89,90.82
5,True,True,True,206,177,85.92
3,True,False,True,6,5,83.33
1,False,False,True,357,242,67.79
0,False,False,False,414,195,47.10
2,True,False,False,7,1,14.29


Le profil présentant le risque le plus élevé n'est pas celui cumulant systématiquement le plus grand nombre de facteurs. Les contrats associant une remise supérieure à 40 % et une remise de courte durée atteignent un taux de départ de 90,8 %, légèrement supérieur au profil intégrant également un service complémentaire (85,9 %). Ce résultat suggère que les conditions promotionnelles constituent les principaux marqueurs du risque de départ à long terme, tandis que la souscription d'un service apporte une information complémentaire mais moins discriminante.

### 11 EXPORTATION DES DONNEES POUR CLIENTS 2026

In [80]:
# ============================================================
# PRÉPARATION DES CONTRATS ACTIFS COMMENCÉS EN 2026
# ============================================================

clients_actifs_2026 = regroupement_centre[
    # Contrats commencés en 2026
    (regroupement_centre["Rent starts on"].dt.year == 2026)

    # Contrats actuellement actifs
    & (regroupement_centre["Contract Status"] == "Active")

    # Location déjà commencée à la date d'observation
    & (regroupement_centre["Rent starts on"] <= date_observation)
].copy()

# Ancienneté du contrat à la date d'observation
clients_actifs_2026["Anciennete_jours"] = (
    date_observation
    - clients_actifs_2026["Rent starts on"]
).dt.days

clients_actifs_2026["Anciennete_mois"] = (
    clients_actifs_2026["Anciennete_jours"] / 30.44
).round(1)

# Déterminer les horizons encore pertinents
clients_actifs_2026["Eligible_score_4_mois"] = (
    clients_actifs_2026["Anciennete_jours"] < 4 * 30.44
)

clients_actifs_2026["Eligible_score_6_mois"] = (
    clients_actifs_2026["Anciennete_jours"] < 6 * 30.44
)

clients_actifs_2026["Eligible_score_12_mois"] = (
    clients_actifs_2026["Anciennete_jours"] < 12 * 30.44
)

# Colonnes conservées dans la future liste confidentielle
colonnes_identification = [
    "Id",
    "Id_centre",
    "Contract Id",
    "Centre",
    "Contract Status",
    "Rent starts on",
    "Anciennete_jours",
    "Anciennete_mois",
    "Area",
    "Insurance Premium",
    "Discount (%)",
    "Discount duration",
    "Service",
    "Eligible_score_4_mois",
    "Eligible_score_6_mois",
    "Eligible_score_12_mois"
]

liste_clients_2026 = clients_actifs_2026[
    colonnes_identification
].copy()

# Contrôles
print("Nombre de contrats actifs commencés en 2026 :", len(liste_clients_2026))

print(
    "Contrats encore concernés par l'horizon de 4 mois :",
    liste_clients_2026["Eligible_score_4_mois"].sum()
)

print(
    "Contrats encore concernés par l'horizon de 6 mois :",
    liste_clients_2026["Eligible_score_6_mois"].sum()
)

print(
    "Contrats encore concernés par l'horizon de 12 mois :",
    liste_clients_2026["Eligible_score_12_mois"].sum()
)


Nombre de contrats actifs commencés en 2026 : 757
Contrats encore concernés par l'horizon de 4 mois : 545
Contrats encore concernés par l'horizon de 6 mois : 718
Contrats encore concernés par l'horizon de 12 mois : 757


In [81]:
# ============================================================
# CALCUL DES SCORES DE DÉPART DES CONTRATS ACTIFS DE 2026
# ============================================================

from sklearn.base import clone

# ------------------------------------------------------------
# 1. Réentraînement définitif sur l'historique disponible
# ------------------------------------------------------------

modele_final_depart_4 = clone(modele_foret_depart_4)
modele_final_depart_6 = clone(modele_foret_depart_6)
modele_final_depart_12 = clone(modele_foret_depart_12)

# Regroupement de l'entraînement 2022-2024 et du test 2025
X_historique_depart_4 = pd.concat(
    [X_train_depart_4, X_test_depart_4],
    axis=0
)

y_historique_depart_4 = pd.concat(
    [y_train_depart_4, y_test_depart_4],
    axis=0
)

X_historique_depart_6 = pd.concat(
    [X_train_depart_6, X_test_depart_6],
    axis=0
)

y_historique_depart_6 = pd.concat(
    [y_train_depart_6, y_test_depart_6],
    axis=0
)

X_historique_depart_12 = pd.concat(
    [X_train_depart_12, X_test_depart_12],
    axis=0
)

y_historique_depart_12 = pd.concat(
    [y_train_depart_12, y_test_depart_12],
    axis=0
)

# Entraînement définitif
modele_final_depart_4.fit(
    X_historique_depart_4,
    y_historique_depart_4
)

modele_final_depart_6.fit(
    X_historique_depart_6,
    y_historique_depart_6
)

modele_final_depart_12.fit(
    X_historique_depart_12,
    y_historique_depart_12
)

print("Les trois modèles définitifs sont entraînés.")

Les trois modèles définitifs sont entraînés.


In [82]:
# ============================================================
# AJOUT DES SCORES AUX CONTRATS ACTIFS DE 2026
# ============================================================

# Variables attendues par chaque modèle
variables_modele_4 = list(
    modele_final_depart_4.feature_names_in_
)

variables_modele_6 = list(
    modele_final_depart_6.feature_names_in_
)

variables_modele_12 = list(
    modele_final_depart_12.feature_names_in_
)

# Initialisation des colonnes
liste_clients_2026["Score_depart_4_mois"] = np.nan
liste_clients_2026["Score_depart_6_mois"] = np.nan
liste_clients_2026["Score_depart_12_mois"] = np.nan


# ------------------------------------------------------------
# SCORE DE DÉPART À 4 MOIS
# ------------------------------------------------------------

index_4_mois = liste_clients_2026.loc[
    liste_clients_2026["Eligible_score_4_mois"]
].index

if len(index_4_mois) > 0:
    liste_clients_2026.loc[
        index_4_mois,
        "Score_depart_4_mois"
    ] = modele_final_depart_4.predict_proba(
        donnees_modelisation.loc[
            index_4_mois,
            variables_modele_4
        ]
    )[:, 1]


# ------------------------------------------------------------
# SCORE DE DÉPART À 6 MOIS
# ------------------------------------------------------------

index_6_mois = liste_clients_2026.loc[
    liste_clients_2026["Eligible_score_6_mois"]
].index

if len(index_6_mois) > 0:
    liste_clients_2026.loc[
        index_6_mois,
        "Score_depart_6_mois"
    ] = modele_final_depart_6.predict_proba(
        donnees_modelisation.loc[
            index_6_mois,
            variables_modele_6
        ]
    )[:, 1]


# ------------------------------------------------------------
# SCORE DE DÉPART À 12 MOIS
# ------------------------------------------------------------

index_12_mois = liste_clients_2026.loc[
    liste_clients_2026["Eligible_score_12_mois"]
].index

if len(index_12_mois) > 0:
    liste_clients_2026.loc[
        index_12_mois,
        "Score_depart_12_mois"
    ] = modele_final_depart_12.predict_proba(
        donnees_modelisation.loc[
            index_12_mois,
            variables_modele_12
        ]
    )[:, 1]


# ------------------------------------------------------------
# CONVERSION DES SCORES EN POURCENTAGES
# ------------------------------------------------------------

colonnes_scores = [
    "Score_depart_4_mois",
    "Score_depart_6_mois",
    "Score_depart_12_mois"
]

liste_clients_2026[colonnes_scores] = (
    liste_clients_2026[colonnes_scores] * 100
).round(2)


# ------------------------------------------------------------
# CONTRÔLE ANONYME DU RÉSULTAT
# Aucun identifiant client n'est affiché
# ------------------------------------------------------------

print(
    "Nombre total de contrats analysés :",
    len(liste_clients_2026)
)

print(
    "Scores à 4 mois calculés :",
    liste_clients_2026[
        "Score_depart_4_mois"
    ].notna().sum()
)

print(
    "Scores à 6 mois calculés :",
    liste_clients_2026[
        "Score_depart_6_mois"
    ].notna().sum()
)

print(
    "Scores à 12 mois calculés :",
    liste_clients_2026[
        "Score_depart_12_mois"
    ].notna().sum()
)

# Statistiques générales sans afficher les clients
resume_scores_2026 = liste_clients_2026[
    [
        "Score_depart_4_mois",
        "Score_depart_6_mois",
        "Score_depart_12_mois"
    ]
].describe().round(2)

display(resume_scores_2026)

Nombre total de contrats analysés : 757
Scores à 4 mois calculés : 545
Scores à 6 mois calculés : 718
Scores à 12 mois calculés : 757


,Score_depart_4_mois,Score_depart_6_mois,Score_depart_12_mois
count,545.00,718.00,757.00
mean,18.28,18.37,22.74
std,13.93,14.12,14.80
min,3.71,2.79,5.54
25%,9.48,9.30,13.80
50%,14.08,14.56,18.62
75%,21.53,21.57,24.30
max,81.24,85.21,88.04


In [83]:
# ============================================================
# CALIBRATION DES NIVEAUX DE RISQUE SUR LE TEST 2025
# ============================================================

def calibrer_niveaux_risque(
    modele,
    X_test,
    y_test,
    horizon
):
    # Probabilités estimées sur les contrats de 2025
    scores = modele.predict_proba(X_test)[:, 1]

    calibration = pd.DataFrame({
        "Score": scores,
        "Depart_observe": y_test.to_numpy()
    })

    # Seuils séparant les contrats en trois groupes de taille proche
    seuil_faible = calibration["Score"].quantile(1 / 3)
    seuil_eleve = calibration["Score"].quantile(2 / 3)

    calibration["Niveau_risque"] = pd.cut(
        calibration["Score"],
        bins=[
            -np.inf,
            seuil_faible,
            seuil_eleve,
            np.inf
        ],
        labels=[
            "Risque faible",
            "Risque modéré",
            "Risque élevé"
        ]
    )

    tableau = (
        calibration
        .groupby(
            "Niveau_risque",
            observed=False
        )
        .agg(
            Nombre_contrats=("Depart_observe", "size"),
            Nombre_departs=("Depart_observe", "sum"),
            Score_minimum=("Score", "min"),
            Score_maximum=("Score", "max"),
            Score_moyen=("Score", "mean"),
            Taux_depart_observe=("Depart_observe", "mean")
        )
        .reset_index()
    )

    # Mise en pourcentage pour la lecture
    for colonne in [
        "Score_minimum",
        "Score_maximum",
        "Score_moyen",
        "Taux_depart_observe"
    ]:
        tableau[colonne] = (
            tableau[colonne] * 100
        ).round(2)

    tableau.insert(0, "Horizon", horizon)

    return tableau, seuil_faible, seuil_eleve


# Calibration à 4 mois
calibration_4, seuil_faible_4, seuil_eleve_4 = (
    calibrer_niveaux_risque(
        modele_foret_depart_4,
        X_test_depart_4,
        y_test_depart_4,
        "4 mois"
    )
)

# Calibration à 6 mois
calibration_6, seuil_faible_6, seuil_eleve_6 = (
    calibrer_niveaux_risque(
        modele_foret_depart_6,
        X_test_depart_6,
        y_test_depart_6,
        "6 mois"
    )
)

# Calibration à 12 mois
calibration_12, seuil_faible_12, seuil_eleve_12 = (
    calibrer_niveaux_risque(
        modele_foret_depart_12,
        X_test_depart_12,
        y_test_depart_12,
        "12 mois"
    )
)

# Tableau global, sans aucune donnée client
tableau_calibration = pd.concat(
    [
        calibration_4,
        calibration_6,
        calibration_12
    ],
    ignore_index=True
)

display(tableau_calibration)

print("\nSeuils calculés sur les contrats de 2025 :")

print(
    "4 mois :",
    round(seuil_faible_4 * 100, 2),
    "% et",
    round(seuil_eleve_4 * 100, 2),
    "%"
)

print(
    "6 mois :",
    round(seuil_faible_6 * 100, 2),
    "% et",
    round(seuil_eleve_6 * 100, 2),
    "%"
)

print(
    "12 mois :",
    round(seuil_faible_12 * 100, 2),
    "% et",
    round(seuil_eleve_12 * 100, 2),
    "%"
)

,Horizon,Niveau_risque,Nombre_contrats,Nombre_departs,Score_minimum,Score_maximum,Score_moyen,Taux_depart_observe
0,4 mois,Risque faible,642,34,4.95,28.89,19.84,5.30
1,4 mois,Risque modéré,639,182,28.89,49.80,37.43,28.48
2,4 mois,Risque élevé,640,458,49.81,84.18,64.40,71.56
3,6 mois,Risque faible,641,35,5.64,29.07,19.56,5.46
4,6 mois,Risque modéré,640,267,29.13,53.01,40.42,41.72
5,6 mois,Risque élevé,640,538,53.01,89.43,67.22,84.06
6,12 mois,Risque faible,364,53,6.80,40.80,21.08,14.56
7,12 mois,Risque modéré,361,302,40.81,62.54,52.12,83.66
8,12 mois,Risque élevé,363,354,62.58,90.85,76.38,97.52



Seuils calculés sur les contrats de 2025 :
4 mois : 28.89 % et 49.8 %
6 mois : 29.07 % et 53.01 %
12 mois : 40.8 % et 62.57 %


In [84]:
# ============================================================
# ATTRIBUTION DES NIVEAUX DE RISQUE AUX CONTRATS DE 2026
# ============================================================

def attribuer_niveau_risque(
    score_pourcentage,
    seuil_faible,
    seuil_eleve
):
    # Aucun niveau lorsque l'horizon n'est plus pertinent
    if pd.isna(score_pourcentage):
        return "Horizon dépassé"

    # Conversion des seuils en pourcentage
    seuil_faible_pct = seuil_faible * 100
    seuil_eleve_pct = seuil_eleve * 100

    if score_pourcentage <= seuil_faible_pct:
        return "Risque faible"

    if score_pourcentage <= seuil_eleve_pct:
        return "Risque modéré"

    return "Risque élevé"


# Niveau à 4 mois
liste_clients_2026["Risque_depart_4_mois"] = (
    liste_clients_2026["Score_depart_4_mois"].apply(
        lambda score: attribuer_niveau_risque(
            score,
            seuil_faible_4,
            seuil_eleve_4
        )
    )
)

# Niveau à 6 mois
liste_clients_2026["Risque_depart_6_mois"] = (
    liste_clients_2026["Score_depart_6_mois"].apply(
        lambda score: attribuer_niveau_risque(
            score,
            seuil_faible_6,
            seuil_eleve_6
        )
    )
)

# Niveau à 12 mois
liste_clients_2026["Risque_depart_12_mois"] = (
    liste_clients_2026["Score_depart_12_mois"].apply(
        lambda score: attribuer_niveau_risque(
            score,
            seuil_faible_12,
            seuil_eleve_12
        )
    )
)

In [85]:
# ============================================================
# CRÉATION D'UNE PRIORITÉ OPÉRATIONNELLE UNIQUE
# ============================================================

def definir_priorite(row):

    # Risque élevé à court terme
    if row["Risque_depart_4_mois"] == "Risque élevé":
        return "1 - Priorité urgente"

    # Risque élevé à 6 mois
    if row["Risque_depart_6_mois"] == "Risque élevé":
        return "2 - Priorité haute"

    # Risque modéré à court terme
    if row["Risque_depart_4_mois"] == "Risque modéré":
        return "2 - Priorité haute"

    # Risque élevé ou modéré à 12 mois
    if (
        row["Risque_depart_12_mois"] == "Risque élevé"
        or row["Risque_depart_6_mois"] == "Risque modéré"
    ):
        return "3 - À anticiper"

    if row["Risque_depart_12_mois"] == "Risque modéré":
        return "4 - Surveillance"

    return "5 - Priorité faible"


liste_clients_2026["Priorite_operationnelle"] = (
    liste_clients_2026.apply(
        definir_priorite,
        axis=1
    )
)

In [86]:
# ============================================================
# CONTRÔLE ANONYME DE LA PRIORISATION
# ============================================================

print("Répartition des risques à 4 mois :")
display(
    liste_clients_2026[
        "Risque_depart_4_mois"
    ].value_counts()
)

print("Répartition des risques à 6 mois :")
display(
    liste_clients_2026[
        "Risque_depart_6_mois"
    ].value_counts()
)

print("Répartition des risques à 12 mois :")
display(
    liste_clients_2026[
        "Risque_depart_12_mois"
    ].value_counts()
)

print("Priorité opérationnelle finale :")
display(
    liste_clients_2026[
        "Priorite_operationnelle"
    ].value_counts().sort_index()
)

Répartition des risques à 4 mois :


Risque_depart_4_mois
Risque faible      466
Horizon dépassé    212
Risque modéré       55
Risque élevé        24
Name: count, dtype: int64

Répartition des risques à 6 mois :


Risque_depart_6_mois
Risque faible      618
Risque modéré       75
Horizon dépassé     39
Risque élevé        25
Name: count, dtype: int64

Répartition des risques à 12 mois :


Risque_depart_12_mois
Risque faible    675
Risque modéré     52
Risque élevé      30
Name: count, dtype: int64

Priorité opérationnelle finale :


Priorite_operationnelle
1 - Priorité urgente     24
2 - Priorité haute       57
3 - À anticiper          24
4 - Surveillance          5
5 - Priorité faible     647
Name: count, dtype: int64

## Synthèse de la modélisation des départs

Trois horizons ont été étudiés afin de distinguer le risque de départ à court et moyen terme : 4, 6 et 12 mois après le début de la location.

### Performances des modèles

Le modèle Random Forest présente les meilleures performances sur les contrats de 2025 :

| Horizon | ROC-AUC | F1-score | Interprétation |
|---|---:|---:|---|
| 4 mois | 0,857 | 0,690 | Bonne capacité de détection à court terme |
| 6 mois | 0,892 | 0,768 | Très bonne distinction des contrats à risque |
| 12 mois | 0,925 | 0,850 | Meilleure performance globale |

Les performances augmentent avec l’horizon. Les données disponibles permettent donc de mieux identifier un profil de location courte que de prévoir précisément un départ imminent.

### Principaux facteurs associés au départ

La prime d’assurance constitue le principal marqueur statistique utilisé par les trois modèles. Elle ne doit cependant pas être interprétée comme la cause du départ : son montant dépend notamment de la couverture choisie et peut également refléter une période tarifaire ou une catégorie de contrat.

La durée et le pourcentage de remise, ainsi que la souscription d’un service complémentaire, apportent également une information utile. L’analyse descriptive montre notamment une fréquence de départ plus élevée parmi les contrats combinant :

- une remise supérieure à 40 % ;
- une remise de courte durée, principalement un ou deux mois ;
- la souscription d’un service complémentaire.

Ces caractéristiques définissent un profil de vigilance commerciale et non un profil certain de départ.

### Application aux contrats de 2026

Les modèles ont été appliqués aux 757 contrats actifs commencés en 2026. Les horizons déjà dépassés ont été exclus afin de ne pas attribuer un score devenu sans objet.

La priorisation obtenue est la suivante :

| Priorité | Nombre de contrats | Action proposée |
|---|---:|---|
| Urgente | 24 | Contact commercial à court terme |
| Haute | 57 | Vigilance renforcée |
| À anticiper | 24 | Préparer une action préventive |
| Surveillance | 5 | Suivi régulier |
| Faible | 647 | Pas d’action spécifique immédiate |

Les 81 contrats classés en priorité urgente ou haute constituent la première liste de contact proposée aux équipes.

### Limites

Les scores constituent une aide à la priorisation et non une certitude individuelle. Ils ne doivent entraîner ni décision automatique ni traitement défavorable du client.

Les seuils ont été calibrés sur les départs observés en 2025. Leur stabilité devra être contrôlée avec les résultats réels de 2026 et 2027. L’ajout de données sur les contacts commerciaux, les motifs de départ, les changements tarifaires, les réclamations et l’utilisation du box permettrait d’améliorer les futurs modèles.

## Limite de la modélisation des impayés

Une modélisation prédictive des impayés avait initialement été envisagée. La population exploitable comprend toutefois seulement **69 situations d’impayé sur 8 566 contrats ayant commencé**, soit **0,81 %** :

- **51 contrats** avec le statut `Access Revoked`, correspondant à une situation potentiellement régularisable ;
- **18 contrats** avec le statut `Over locked`, correspondant à une situation plus critique.

Ce faible nombre de cas positifs ne permet pas de construire un modèle suffisamment robuste et stable. Un modèle pourrait obtenir une accuracy artificiellement élevée en prédisant presque systématiquement l’absence d’impayé, sans identifier correctement les situations à risque.

L’analyse des impayés est donc limitée, dans cette version, à un suivi descriptif et opérationnel. La construction d’un futur modèle nécessiterait notamment :

- l’historique détaillé des factures ;
- les dates d’échéance ;
- les montants dus et réglés ;
- le nombre et la durée des retards ;
- les relances effectuées ;
- les régularisations ;
- les incidents de paiement antérieurs.

Cette décision évite de présenter à l’entreprise un modèle insuffisamment fiable. Les 69 situations identifiées restent disponibles pour organiser et prioriser les actions de recouvrement.